# Rotating Amorphous Dipole — 2D FDTD Radiation Study (Version 4)

**Author:** Joshua Clavecillas  
**Version:** 4 (cleaned, improved, partial quantitative validation)  
**Date:** April 2026

---

## Project Goal

Simulate electromagnetic radiation from a **rotating amorphous dipole** source using a 2D TMz-style FDTD-like Maxwell solver. Validate the radiation using:

- Dipole moment tracking  
- Field snapshots  
- Poynting vector flow  
- Ring flux vs time  
- Angular intensity pattern  
- Wave speed measurement  
- Resolution convergence test  
- Frequency scaling study  

This notebook is designed to run **top-to-bottom** on a laptop in a reasonable time. Heavy parameter sweeps are guarded behind optional flags.

## Section 1 — Physical Model and Limitations

### Physical Model

A **rotating amorphous dipole** is a charge distribution consisting of two opposite-sign Gaussian blobs whose centers orbit the origin at angular velocity $\omega$. As the charge distribution rotates, it drives time-varying current densities $J_x, J_y$, which in turn radiate electromagnetic energy.

The electromagnetic fields $(E_x, E_y, B_z)$ are evolved using the **2D TMz Maxwell equations**:

$$\frac{\partial B_z}{\partial t} = -\left(\frac{\partial E_y}{\partial x} - \frac{\partial E_x}{\partial y}\right)$$

$$\frac{\partial E_x}{\partial t} = c^2 \frac{\partial B_z}{\partial y} - \frac{J_x}{\varepsilon_0}$$

$$\frac{\partial E_y}{\partial t} = -c^2 \frac{\partial B_z}{\partial x} - \frac{J_y}{\varepsilon_0}$$

### Honest Limitations

| Limitation | Impact |
|---|---|
| **2D geometry, not 3D** | Field structure and radiation patterns differ from 3D; $\omega^4$ law does not apply exactly |
| **Collocated grid** | All fields on same grid points; not a staggered Yee lattice; dispersion is worse |
| **Prescribed source** | $\rho$ and $J$ are set analytically; not self-consistent charge dynamics |
| **Gauss's law not enforced** | $\nabla \cdot E \neq \rho/\varepsilon_0$ exactly; reported as diagnostic |
| **Mur ABC + damping layer** | Not a perfect PML; reflections possible from boundaries |
| **Finite source size** | Near-field effects, finite wavelength-to-source-size ratio |
| **Finite domain** | Boundary reflections contaminate late-time results |

This simulation is best described as a **qualitative-to-semiquantitative 2D FDTD-like model**, not a quantitatively exact 3D dipole radiation solver.

## Section 2 — Numerical Method

### TMz FDTD-Like Update (Collocated Grid)

This notebook uses a **2D TMz-style collocated-grid Maxwell solver**. All three field components $(E_x, E_y, B_z)$ are stored at the **same grid points** (not staggered as in a proper Yee lattice). This simplifies the implementation at the cost of slightly worse numerical dispersion.

The time-stepping is a simple **forward Euler** / **leapfrog-style** scheme:

1. Update $B_z$ using the curl of $E$  
2. Update $E_x$, $E_y$ using the curl of $B_z$ and injected sources  
3. Apply absorbing boundary conditions (Mur ABC + damping layer)

### CFL Stability Condition

For a 2D wave equation on a square grid ($\Delta x = \Delta y$), the CFL stability condition is:

$$\text{CFL} = \frac{c \, \Delta t}{\Delta x} \leq \frac{1}{\sqrt{2}} \approx 0.707$$

We use `cfl_safety = 0.38` for a comfortable safety margin.

### Boundary Conditions

- **Mur first-order ABC**: applied at all four edges to absorb outgoing waves  
- **Absorbing damping layer**: a smooth mask $\sigma(x,y) \in [0,1]$ applied multiplicatively to fields near the boundary each timestep — a simple approximation, not a full PML

## Section 3 — Imports, Grid, Constants, and CFL Settings

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import time
import json
from pathlib import Path
from IPython.display import display
import pandas as pd

# ── Normalized units ────────────────────────────────────────────────────────
epsilon0 = 1.0
mu0      = 1.0
c        = 1.0

np.set_printoptions(precision=6, suppress=True)

# ── Main simulation grid ─────────────────────────────────────────────────────
Nx_main  = 400
Ny_main  = 400
Lx_main  = 8
Ly_main  = 8

dx_main  = Lx_main / Nx_main
dy_main  = Ly_main / Ny_main

cfl_safety = 0.2
dt_main    = cfl_safety * min(dx_main, dy_main) / c

cfl_number   = c * dt_main / dx_main
cfl_limit_2d = 1.0 / np.sqrt(2.0)
assert cfl_number <= cfl_limit_2d, f"CFL violated: {cfl_number:.4f} > {cfl_limit_2d:.4f}"

# Build grid arrays
x_main = np.linspace(-Lx_main/2, Lx_main/2, Nx_main, endpoint=False)
y_main = np.linspace(-Ly_main/2, Ly_main/2, Ny_main, endpoint=False)
X_main, Y_main = np.meshgrid(x_main, y_main, indexing='xy')

print("── Grid & CFL Summary ─────────────────────────────────")
print(f"  Nx_main = {Nx_main}, Ny_main = {Ny_main}")
print(f"  Lx_main = {Lx_main}, Ly_main = {Ly_main}")
print(f"  dx_main = {dx_main:.6f}, dy_main = {dy_main:.6f}")
print(f"  dt_main = {dt_main:.6f}")
print(f"  CFL     = {cfl_number:.4f}  (limit ≤ {cfl_limit_2d:.4f})  ✓ stable")
print("───────────────────────────────────────────────────────")

### JC: This IPYNB is the fifth iteration so im just gonna reiterate before what I originially thought I needed to start with first and evolve the ipynb. 

### First I had to make a grid with divisions to plot out the propagating fields and charges. The more divisions would mean higher resolution at the cost of run time. Because this is a time dependent wave propagation simulation an Explicit method would be the most reasonable to implement. However this comes at a cost of CFL condition. 

## Section 4 — Source Construction: Rotating Amorphous Dipole

The charge density $\rho(x,y,t)$ consists of two **smooth Gaussian blobs** with opposite sign. Their centers rotate counter-clockwise at angular velocity $\omega$:

$$\mathbf{r}_{\pm}(t) = \pm\frac{d}{2}\begin{pmatrix}\cos(\omega t)\\\sin(\omega t)\end{pmatrix}$$

Each blob is slightly *amorphous* — a small angular distortion is added to make the source non-trivially shaped while remaining well-behaved. The blobs are normalized so that total charge is exactly zero at all times.

In [ ]:
# ── Source parameters (shared across all sections) ───────────────────────────
omega_main  = 1.5  # angular frequency of rotation
Q_main      = 10.0   # charge magnitude
sep_main    = 0.8    # separation between charge centers
a_main      = 0.2    # blob radius (Gaussian-mixture envelope)
b_main      = 0.4    # kept for API compatibility (unused by mixture blob)
phase_main  = 0    # legacy parameter, unused (kept so old call signatures work)

r_obs_main  = 3.7    # observation ring radius
periods_main = 2.0  # number of rotation periods to simulate

# ── Source type selector ──────────────────────────────────────────────────────
# Options:
#   'amorphous'  – rotating dipole made of two amorphous Gaussian-mixture blobs
#                  (the only blob model in this notebook)
#   'circular'   – N point charges (narrow Gaussians) on a ring; alternating ±Q
SOURCE_TYPE   = 'circular'
N_ring        = 2             # number of charges on the ring (circular mode; must be even)
ring_radius   = sep_main / 10 # radius of the charge ring (circular mode)
sigma_pt      = 0.1           # width of each point-charge Gaussian (circular mode)

# Mixture-blob shape controls (deterministic given these seeds)
N_LUMPS       = 5            # Gaussian sub-lumps per blob
SEED_PLUS     = 1             # RNG seed for the +Q blob
SEED_MINUS    = 2             # RNG seed for the −Q blob


# ── Amorphous Gaussian-mixture blob ──────────────────────────────────────────
def amorphous_blob(X, Y, center=(0.0, 0.0), total_charge=1.0,
                   radius=0.5, n_lumps=12, seed=0):
    """
    Smooth amorphous charge density built from a mixture of randomly placed,
    anisotropic Gaussian sub-lumps, multiplied by an outer Gaussian envelope and
    normalized so that ∫ρ dx dy = total_charge.

    Reproducible: random sub-lumps are drawn from np.random.default_rng(seed).
    """
    rng = np.random.default_rng(seed)
    x0, y0 = center

    # Rotation-invariant grid spacing (works even when X, Y are rotated).
    dx = float(np.hypot(X[0, 1] - X[0, 0], Y[0, 1] - Y[0, 0]))
    dy = float(np.hypot(X[1, 0] - X[0, 0], Y[1, 0] - Y[0, 0]))

    rho = np.zeros_like(X, dtype=float)
    for _ in range(n_lumps):
        r_k    = radius * np.sqrt(rng.uniform(0.0, 1.0))   # uniform area in disk
        phi_k  = rng.uniform(0.0, 2.0 * np.pi)
        cx_k   = x0 + r_k * np.cos(phi_k)
        cy_k   = y0 + r_k * np.sin(phi_k)
        sx_k   = radius * rng.uniform(0.15, 0.45)
        sy_k   = radius * rng.uniform(0.15, 0.45)
        theta  = rng.uniform(0.0, np.pi)
        ct, st = np.cos(theta), np.sin(theta)
        Xc =  (X - cx_k) * ct + (Y - cy_k) * st
        Yc = -(X - cx_k) * st + (Y - cy_k) * ct
        amp_k = rng.uniform(0.4, 1.0)
        rho += amp_k * np.exp(-0.5 * (Xc**2 / sx_k**2 + Yc**2 / sy_k**2))

    # Smooth outer envelope → soft edges (avoids high-k FDTD noise).
    envelope = np.exp(-0.5 * ((X - x0)**2 + (Y - y0)**2) / radius**2)
    rho *= envelope

    q_raw = np.sum(rho) * dx * dy
    if q_raw != 0.0:
        rho *= (total_charge / q_raw)
    return rho


def rotate_coordinates(X, Y, angle):
    """Rigid coordinate rotation: Xr = c·X + s·Y,  Yr = -s·X + c·Y."""
    c, s = np.cos(angle), np.sin(angle)
    return c * X + s * Y, -s * X + c * Y


def make_amorphous_dipole(X, Y, angle, Q, d, a,
                          n_lumps=15, seed_plus=1, seed_minus=2):
    """
    Rotating dipole = two amorphous Gaussian-mixture blobs of opposite sign,
    rigidly rotated by `angle` (no re-randomization in time).

    Returns
    -------
    rho : 2D array, total signed charge density (rho_plus + rho_minus).
    """
    Xr, Yr = rotate_coordinates(X, Y, angle)
    rho_plus  = amorphous_blob(Xr, Yr, center=(+d/2, 0.0),
                               total_charge=+Q, radius=a,
                               n_lumps=n_lumps, seed=seed_plus)
    rho_minus = amorphous_blob(Xr, Yr, center=(-d/2, 0.0),
                               total_charge=-Q, radius=a,
                               n_lumps=n_lumps, seed=seed_minus)
    return rho_plus + rho_minus


# ── Diagnostic helpers ────────────────────────────────────────────────────────
def total_charge(rho, dx, dy):
    """Total integrated charge ∫ρ dx dy."""
    return np.sum(rho) * dx * dy


def dipole_moment(rho, X, Y, dx, dy):
    """First moment of charge density: (px, py) = ∫ρ (x, y) dx dy."""
    px = np.sum(rho * X) * dx * dy
    py = np.sum(rho * Y) * dx * dy
    return px, py


# ── Circular ring source (kept; alternative to the mixture blob) ─────────────
def circular_ring_rho(X, Y, angle, Q, ring_r, sigma, N, dx, dy):
    """N point Gaussians evenly spaced on a ring, alternating ±Q (net = 0)."""
    rho = np.zeros_like(X)
    charges = []
    for k in range(N):
        phi  = angle + 2 * np.pi * k / N
        xk   = ring_r * np.cos(phi)
        yk   = ring_r * np.sin(phi)
        sign = +1 if k % 2 == 0 else -1
        blob = np.exp(-((X - xk)**2 + (Y - yk)**2) / (2 * sigma**2))
        charges.append((blob, sign))
    for blob, sign in charges:
        q_raw = np.sum(blob) * dx * dy
        if q_raw > 0:
            rho += sign * blob * (Q / q_raw)
    return rho


# ── Charge-density dispatcher ────────────────────────────────────────────────
def rotating_rho(X, Y, t, omega, Q=1.0, sep=1.0, a=0.35, b=0.24, phase=0.6,
                 dx=1.0, dy=1.0, source_type='amorphous',
                 n_ring=8, ring_r=None, sigma_pt=0.08,
                 n_lumps=15, seed_plus=1, seed_minus=2):
    """
    source_type='amorphous'  → rotating dipole built from two Gaussian-mixture blobs
    source_type='circular'   → N alternating ±Q point-Gaussians on a rotating ring
    """
    angle = omega * t

    if source_type == 'circular':
        if ring_r is None:
            ring_r = sep / 2
        rho_total = circular_ring_rho(X, Y, angle, Q, ring_r, sigma_pt, n_ring, dx, dy)
        rho_pos = np.where(rho_total > 0, rho_total, 0.0)
        rho_neg = np.where(rho_total < 0, rho_total, 0.0)
        return rho_total, rho_pos, rho_neg

    # default: amorphous Gaussian-mixture dipole
    rho = make_amorphous_dipole(X, Y, angle, Q=Q, d=sep, a=max(a, b),
                                n_lumps=n_lumps,
                                seed_plus=seed_plus, seed_minus=seed_minus)
    rho_pos = np.where(rho > 0, rho, 0.0)
    rho_neg = np.where(rho < 0, rho, 0.0)
    return rho, rho_pos, rho_neg


def compute_dipole_moment(rho, X, Y, dx, dy):
    """Alias kept for backward compatibility with downstream cells."""
    return dipole_moment(rho, X, Y, dx, dy)


# ── Source position helper (for plot overlays) ────────────────────────────────
def source_positions(t, omega, sep, source_type='amorphous',
                     n_ring=8, ring_r=None, phase=0.0):
    """Return (xs, ys, signs) for visualization overlays."""
    angle = omega * t
    if source_type == 'circular':
        if ring_r is None:
            ring_r = sep / 2
        ks    = np.arange(n_ring)
        phis  = angle + 2*np.pi*ks/n_ring
        xs    = ring_r * np.cos(phis)
        ys    = ring_r * np.sin(phis)
        signs = np.where(ks % 2 == 0, +1, -1)
        return xs, ys, signs
    # amorphous: two blob centers rotated by ωt
    ca, sa = np.cos(angle), np.sin(angle)
    xs    = np.array([ (sep/2) * ca, -(sep/2) * ca])
    ys    = np.array([ (sep/2) * sa, -(sep/2) * sa])
    signs = np.array([+1, -1])
    return xs, ys, signs


def overlay_source(ax, t, omega=None, sep=None, source_type=None,
                   n_ring=None, ring_r=None, label_t=True,
                   marker_size=70, alpha=0.95, show_orbit=True):
    """Draw markers on `ax` showing source positions at time t."""
    if omega       is None: omega       = omega_main
    if sep         is None: sep         = sep_main
    if source_type is None: source_type = SOURCE_TYPE
    if n_ring      is None: n_ring      = N_ring
    if ring_r      is None: ring_r      = ring_radius

    xs, ys, signs = source_positions(t, omega, sep, source_type,
                                     n_ring=n_ring, ring_r=ring_r)
    pos = signs > 0
    neg = ~pos
    ax.scatter(xs[pos], ys[pos], s=marker_size, marker='o',
               facecolors='none', edgecolors='red',  lw=1.6,
               alpha=alpha, zorder=5, label='+ source')
    ax.scatter(xs[neg], ys[neg], s=marker_size, marker='o',
               facecolors='none', edgecolors='blue', lw=1.6,
               alpha=alpha, zorder=5, label='− source')
    if show_orbit:
        r_orbit = (ring_r if (source_type == 'circular' and ring_r is not None)
                   else sep/2)
        th = np.linspace(0, 2*np.pi, 200)
        ax.plot(r_orbit*np.cos(th), r_orbit*np.sin(th),
                color='lime', lw=0.8, ls=':', alpha=0.55, zorder=4)
    if label_t:
        ax.text(0.02, 0.98, f't = {t:.3f}', transform=ax.transAxes,
                ha='left', va='top', fontsize=9, color='white',
                bbox=dict(facecolor='black', alpha=0.45, pad=2,
                          edgecolor='none'))


# ── Plot initial charge density ───────────────────────────────────────────────
rho0, rho_p0, rho_n0 = rotating_rho(
    X_main, Y_main, t=0.0, omega=omega_main,
    Q=Q_main, sep=sep_main, a=a_main, b=b_main, phase=phase_main,
    dx=dx_main, dy=dy_main,
    source_type=SOURCE_TYPE, n_ring=N_ring, ring_r=ring_radius, sigma_pt=sigma_pt,
    n_lumps=N_LUMPS, seed_plus=SEED_PLUS, seed_minus=SEED_MINUS,
)

print(f"Source type      : {SOURCE_TYPE}")
print(f"Net charge at t=0: {np.sum(rho0)*dx_main*dy_main:.2e}  (should be ~0)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
vmax = np.abs(rho0).max()
im0 = axes[0].imshow(rho0, origin='lower', cmap='coolwarm', vmin=-vmax, vmax=vmax,
                     extent=[-Lx_main/2, Lx_main/2, -Ly_main/2, Ly_main/2])
plt.colorbar(im0, ax=axes[0], fraction=0.046, label=r'$\rho$')
overlay_source(axes[0], t=0.0)
axes[0].legend(loc='upper right', fontsize=8)
axes[0].set_title('Initial charge density $\\rho(x,y,t=0)$')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

# Phase portrait of dipole moment over one period
T_main = 2.0 * np.pi / omega_main
t_demo = np.linspace(0, T_main, 200)
px_demo, py_demo = [], []
for td in t_demo:
    rho_td, *_ = rotating_rho(X_main, Y_main, t=td, omega=omega_main,
                               Q=Q_main, sep=sep_main, a=a_main, b=b_main, phase=phase_main,
                               dx=dx_main, dy=dy_main,
                               source_type=SOURCE_TYPE, n_ring=N_ring,
                               ring_r=ring_radius, sigma_pt=sigma_pt,
                               n_lumps=N_LUMPS,
                               seed_plus=SEED_PLUS, seed_minus=SEED_MINUS)
    pxd, pyd = dipole_moment(rho_td, X_main, Y_main, dx_main, dy_main)
    px_demo.append(pxd); py_demo.append(pyd)

axes[1].plot(px_demo, py_demo, lw=1.5, color='tab:blue')
axes[1].set_aspect('equal')
axes[1].set_title('Dipole phase portrait $p_y$ vs $p_x$ (one period)')
axes[1].set_xlabel('$p_x$'); axes[1].set_ylabel('$p_y$')
axes[1].grid(alpha=0.3)

src_label = ('Rotating Amorphous Gaussian-Mixture Dipole'
             if SOURCE_TYPE == 'amorphous'
             else f'Circular Ring ({N_ring} charges, r={ring_radius})')
plt.suptitle(f'Source: {src_label}', fontsize=13)
plt.tight_layout()
plt.show()


### JC: This section initializes the charge densities and gives it its form, It also sets the parameters for the main solution, being how fast the charges rotate, at what radius is the flux being observed at, and how many rotations does the dipole do. One thing to note is that 
## $$ v = \omega y$$
### So the product of the radius times the omega parameter must be less than c (which is normalized to 1) in order for this code to remain physical, or else these dipoles will rotate faster than the waves can propagate, Ie they can spin faster than the speed of light.

### Next part is to compute the rotation of the particles, which will give us the current. Which also tells us later how the B field will act.


## Section 5 — Current Density and Continuity Check

The current density of a rigidly rotating charge distribution is:

$$J_x = \rho \cdot v_x = \rho \cdot (-\omega y), \quad J_y = \rho \cdot v_y = \rho \cdot (+\omega x)$$

**Continuity diagnostic:** We check $\partial_t \rho + \nabla \cdot \mathbf{J} \approx 0$ using centered finite differences. The normalized residual measures how well the prescribed source satisfies local charge conservation.

In [ ]:
# ── Finite-difference helpers ─────────────────────────────────────────────────
def ddx(F, dx):
    """Centered finite difference in x (non-periodic, one-sided at edges)."""
    dF = np.empty_like(F)
    dF[:, 1:-1] = (F[:, 2:] - F[:, :-2]) / (2*dx)
    dF[:, 0]    = (F[:, 1] - F[:, 0]) / dx
    dF[:, -1]   = (F[:, -1] - F[:, -2]) / dx
    return dF


def ddy(F, dy):
    """Centered finite difference in y (non-periodic, one-sided at edges)."""
    dF = np.empty_like(F)
    dF[1:-1, :] = (F[2:, :] - F[:-2, :]) / (2*dy)
    dF[0, :]    = (F[1, :] - F[0, :]) / dy
    dF[-1, :]   = (F[-1, :] - F[-2, :]) / dy
    return dF


def divergence2d(Fx, Fy, dx, dy):
    return ddx(Fx, dx) + ddy(Fy, dy)


# ── Current density ───────────────────────────────────────────────────────────
def compute_current(rho, X, Y, omega):
    """Rigid rotation current: J = rho * v, v = omega x r_perp."""
    Jx = rho * (-omega * Y)
    Jy = rho * ( omega * X)
    return Jx, Jy


# ── Continuity residual at a single time ─────────────────────────────────────
def continuity_residual(X, Y, t, dt, omega, Q, sep, a, b, phase, dx, dy):
    """Returns (residual_rms, normalized_residual_rms)."""
    rho_p, *_ = rotating_rho(X, Y, t-dt, omega, Q, sep, a, b, phase, dx, dy)
    rho_0, *_ = rotating_rho(X, Y, t,    omega, Q, sep, a, b, phase, dx, dy)
    rho_n, *_ = rotating_rho(X, Y, t+dt, omega, Q, sep, a, b, phase, dx, dy)

    Jx, Jy = compute_current(rho_0, X, Y, omega)
    drho_dt = (rho_n - rho_p) / (2.0 * dt)
    divJ    = divergence2d(Jx, Jy, dx, dy)
    residual = drho_dt + divJ

    core = (slice(2, -2), slice(2, -2))
    rms  = np.sqrt(np.mean(residual[core]**2))
    denom = (np.sqrt(np.mean(drho_dt[core]**2))
             + np.sqrt(np.mean(divJ[core]**2)) + 1e-15)
    return rms, rms / denom


# ── Demo: check continuity at t=0 ────────────────────────────────────────────
rms0, norm0 = continuity_residual(
    X_main, Y_main, t=0.5*T_main, dt=dt_main,
    omega=omega_main, Q=Q_main, sep=sep_main, a=a_main, b=b_main,
    phase=phase_main, dx=dx_main, dy=dy_main
)
print(f"Continuity residual at t=T/2: RMS = {rms0:.4e}, normalized = {norm0:.4e}")
print("A normalized value ≪ 1 means ∂ρ/∂t ≈ -∇·J is well satisfied.")

### JC: This computes moving charges which is what we need for the b field. The continuity residual is quite low for how much charge we are moving, So i believe we can move onto the next step which is actually solving the E field and b field.

### Previous iterations has done a euler time stepping, however we have learned that Euler time stepping takes quite a bit of time and is a first order technique that doesnt approximate waves too well. A better alternative is Leapfrog time stepping where it computes one part first (Electric field) and then calculates the next part (B field and current) at the half steps

## Section 6 — FDTD Solver (2D Yee / Leapfrog)

This project now uses a **2D TEz Yee-staggered FDTD solver with leapfrog time stepping**. The three field components live on **different sub-grids** of the same cell:

| Field | Storage | Native shape | Time level |
|---|---|---|---|
| $B_z$ | cell **centers** | $(N_y,\,N_x)$ | half-integer $n+\tfrac{1}{2}$ |
| $E_x$ | horizontal **edges** ($y$-faces) | $(N_y+1,\,N_x)$ | integer $n$ |
| $E_y$ | vertical **edges** ($x$-faces) | $(N_y,\,N_x+1)$ | integer $n$ |

The leapfrog update per step (with $\varepsilon_0 = \mu_0 = c = 1$) is

$$
\begin{aligned}
B_z^{\,n+\tfrac{1}{2}} &= B_z^{\,n-\tfrac{1}{2}} \;-\; \Delta t\,\bigl[\partial_x E_y^{\,n} - \partial_y E_x^{\,n}\bigr], \\[4pt]
E_x^{\,n+1} &= E_x^{\,n} \;+\; \Delta t\,\Bigl[\, c^{2}\,\partial_y B_z^{\,n+\tfrac{1}{2}} \;-\; J_x/\varepsilon_0 \,\Bigr], \\[4pt]
E_y^{\,n+1} &= E_y^{\,n} \;+\; \Delta t\,\Bigl[\!-c^{2}\,\partial_x B_z^{\,n+\tfrac{1}{2}} \;-\; J_y/\varepsilon_0 \,\Bigr],
\end{aligned}
$$

where each spatial derivative is the natural one-cell centered difference between its **two flanking Yee nodes** (no extra interpolation). The current density is sampled at the native location of each $E$ component:

- $J_x$ is evaluated on the $E_x$ edge grid,
- $J_y$ is evaluated on the $E_y$ edge grid,

so the source coupling is consistent with the staggering. The boundary treatment (chosen by `boundary_method`) is applied to the **tangential** electric components — Mur 1st-order ABC for `"mur"`, hard-wall PEC ($E_\parallel = 0$) for `"none"`/`"damping"`, plus the optional multiplicative damping mask.

### Why Yee + leapfrog

- The discrete spatial operators satisfy $\nabla\!\cdot(\nabla\times \cdot)\equiv 0$ exactly, so $\partial_t \nabla\!\cdot B_z = 0$ to machine precision (no spurious magnetic monopoles).
- Time-staggering of $E$ and $B$ is **second-order accurate in time** without any extra cost; a collocated simultaneous-time update is effectively only first order on the fast modes.
- Numerical dispersion is reduced compared with a collocated grid at the same $\Delta x$.

### CFL stability

For the 2D Yee scheme on a uniform grid the CFL condition is

$$\Delta t \;\le\; \frac{1}{c}\,\Bigl(\tfrac{1}{\Delta x^{2}}+\tfrac{1}{\Delta y^{2}}\Bigr)^{-1/2}, \qquad
\text{CFL}\;\equiv\; c\,\Delta t\sqrt{\tfrac{1}{\Delta x^{2}}+\tfrac{1}{\Delta y^{2}}} \;\le\; 1.$$

The next cell prints this number and warns if it exceeds 1; the solver also re-checks it on every call.

### Diagnostic outputs

For compatibility with the rest of the notebook, `run_fdtd` averages $E_x, E_y$ to **cell centers** before returning, so all downstream analysis cells (snapshots, Poynting, ring flux, dipole moment, continuity / Gauss) work unchanged. The native staggered arrays are also exposed under `Ex_yee`, `Ey_yee`, `Bz_yee` if you want to inspect them directly.


In [ ]:
# ── Core FDTD runner (Yee / leapfrog) ────────────────────────────────────────
def run_fdtd(
    X, Y, dx, dy, dt,
    nsteps=1400, omega=2.0,
    Q=1.0, sep=1.0, a=0.35, b=0.24, phase=0.6,
    r_obs=3.2, ntheta_ring=180,
    save_interval=10, store_frames=True,
    continuity_interval=120,
    extra_r_obs=None,
    extra_rings=None,
    abs_width=1.2, abs_strength=0.018,
    boundary_method="mur",       # "mur" | "damping" | "mur+damping" | "none"
    progress_ticks=False, label='FDTD-Yee',
    source_type='amorphous',
    n_ring=8, ring_r=None, sigma_pt=0.08,
):
    """
    2D TEz FDTD solver on a **Yee staggered grid** with **leapfrog** time stepping.

    Spatial layout (cell (i, j) has center (X[j, i], Y[j, i]); shape (Ny, Nx)):

        Bz[j, i]      at cell centers              shape (Ny,   Nx)
        Ex[j, i]      on horizontal edges          shape (Ny+1, Nx)
                       y_Ex[j] = y_Bz[0] - dy/2 + j·dy
        Ey[j, i]      on vertical edges            shape (Ny,   Nx+1)
                       x_Ey[i] = x_Bz[0] - dx/2 + i·dx

    Time leapfrog (with E known at integer steps n, B at half-integer n±1/2):

        Bz^{n+1/2} = Bz^{n-1/2} - Δt [ ∂_x Eyⁿ - ∂_y Exⁿ ]      (curl on cell)
        Exⁿ⁺¹       = Exⁿ + Δt [  c² ∂_y Bz^{n+1/2} - Jx/ε₀ ]    (∂_y across Ex edge)
        Eyⁿ⁺¹       = Eyⁿ + Δt [ -c² ∂_x Bz^{n+1/2} - Jy/ε₀ ]    (∂_x across Ey edge)

    Boundary treatment (chosen by ``boundary_method``):
        "mur"          Mur 1st-order ABC on tangential E (Ex top/bottom,
                        Ey left/right).  Default — recommended.
        "damping"      Soft PEC (tangential E = 0) + multiplicative damping
                        mask on all three native arrays.
                        ⚠ artificially lossy — comparison only.
        "mur+damping"  Mur ABC + damping mask (legacy v4).
        "none"         Hard PEC walls (tangential E = 0).  Stability /
                        reflection check; no absorption.

    Public outputs (Ex, Ey, Bz, frames, flux, energy) use fields **interpolated
    to the cell-center grid** (X, Y) so the downstream notebook cells continue
    to work without modification.  The native staggered arrays are also exposed
    as ``Ex_yee``, ``Ey_yee``, ``Bz_yee``.
    """
    if boundary_method not in VALID_BOUNDARY_METHODS:
        raise ValueError(f"boundary_method={boundary_method!r} "
                          f"not in {VALID_BOUNDARY_METHODS}")
    Ny, Nx = X.shape

    # ── CFL ─────────────────────────────────────────────────────────────────
    cfl_ratio     = c * dt / dx
    cfl_lim       = 1.0 / np.sqrt(2.0)
    cfl_number_2d = c * dt * np.sqrt(1.0/dx**2 + 1.0/dy**2)
    if cfl_ratio > cfl_lim + 1e-8:
        raise ValueError(f'CFL violated: {cfl_ratio:.4f} > {cfl_lim:.4f}')
    if cfl_number_2d > 1.0:
        print(f"⚠️  CFL number = {cfl_number_2d:.4f} > 1 — Yee solver will be unstable.")

    # ── Internal staggered coordinates ──────────────────────────────────────
    x_bz = X[0, :]
    y_bz = Y[:, 0]
    y_ex = np.concatenate(([y_bz[0] - dy/2.0], y_bz + dy/2.0))   # length Ny+1
    x_ey = np.concatenate(([x_bz[0] - dx/2.0], x_bz + dx/2.0))   # length Nx+1
    X_ex, Y_ex = np.meshgrid(x_bz, y_ex, indexing='xy')          # (Ny+1, Nx)
    X_ey, Y_ey = np.meshgrid(x_ey, y_bz, indexing='xy')          # (Ny,   Nx+1)

    # ── Yee field arrays (zero initial conditions) ──────────────────────────
    Ex = np.zeros((Ny + 1, Nx))
    Ey = np.zeros((Ny,     Nx + 1))
    Bz = np.zeros((Ny,     Nx))

    # ── Damping masks on each native grid ───────────────────────────────────
    Lx_loc = dx * Nx
    Ly_loc = dy * Ny
    mask_bz = make_absorbing_mask(X,    Y,    Lx_loc, Ly_loc,
                                   width=abs_width, strength=abs_strength)
    mask_ex = make_absorbing_mask(X_ex, Y_ex, Lx_loc, Ly_loc,
                                   width=abs_width, strength=abs_strength)
    mask_ey = make_absorbing_mask(X_ey, Y_ey, Lx_loc, Ly_loc,
                                   width=abs_width, strength=abs_strength)

    src_kwargs = dict(source_type=source_type, n_ring=n_ring,
                      ring_r=ring_r, sigma_pt=sigma_pt)

    # ── Output time series ──────────────────────────────────────────────────
    t_arr       = np.arange(nsteps) * dt
    px_arr      = np.zeros(nsteps)
    py_arr      = np.zeros(nsteps)
    flux_signed = np.zeros(nsteps)
    flux_out    = np.zeros(nsteps)
    energy_hist = np.zeros(nsteps)

    flux_r2 = np.zeros(nsteps) if extra_r_obs is not None else None
    extra_rings_list = list(extra_rings) if extra_rings else []
    flux_extras = {float(r_): np.zeros(nsteps) for r_ in extra_rings_list}

    diag_t         = []
    cont_rms_list  = []
    cont_norm_list = []
    gauss_rms_list = []

    frames_Bz  = []
    frames_rho = []
    frames_Ex  = []
    frames_Ey  = []
    frame_steps = []

    tick_interval = max(1, nsteps // 40)
    t0 = time.perf_counter()
    if progress_ticks:
        print(f'{label} [{boundary_method}]: ', end='', flush=True)

    rho_bz = None

    # Mur coefficients (tangential E component sees the wall in its OTHER axis)
    alpha_y = (c*dt - dy) / (c*dt + dy)   # for Ex on top/bottom walls
    alpha_x = (c*dt - dx) / (c*dt + dx)   # for Ey on left/right walls

    for n in range(nsteps):
        t = t_arr[n]

        # ── Source sampling at native E-component locations ─────────────────
        rho_bz, _, _ = rotating_rho(X,    Y,    t=t, omega=omega, Q=Q,
                                     sep=sep, a=a, b=b, phase=phase,
                                     dx=dx, dy=dy, **src_kwargs)
        rho_ex, _, _ = rotating_rho(X_ex, Y_ex, t=t, omega=omega, Q=Q,
                                     sep=sep, a=a, b=b, phase=phase,
                                     dx=dx, dy=dy, **src_kwargs)
        rho_ey, _, _ = rotating_rho(X_ey, Y_ey, t=t, omega=omega, Q=Q,
                                     sep=sep, a=a, b=b, phase=phase,
                                     dx=dx, dy=dy, **src_kwargs)
        # Rigid rotation current J = ρ · (ω × r)
        Jx_at_Ex = rho_ex * (-omega * Y_ex)
        Jy_at_Ey = rho_ey * ( omega * X_ey)

        # ── Step 1: advance Bz (n-1/2 → n+1/2) using Eⁿ ─────────────────────
        # curl_E[j, i] = (Ey[j, i+1] - Ey[j, i])/dx - (Ex[j+1, i] - Ex[j, i])/dy
        curl_E = ((Ey[:, 1:] - Ey[:, :-1]) / dx
                  - (Ex[1:, :] - Ex[:-1, :]) / dy)
        Bz -= dt * curl_E

        if boundary_method in ("damping", "mur+damping"):
            Bz *= mask_bz

        # ── Step 2: advance Ex, Ey (n → n+1) using Bz^{n+1/2} ───────────────
        Ex_prev = Ex.copy()
        Ey_prev = Ey.copy()

        # Ex interior (j = 1..Ny-1)
        Ex[1:-1, :] += dt * (
            (c**2) * (Bz[1:, :] - Bz[:-1, :]) / dy
            - Jx_at_Ex[1:-1, :] / epsilon0
        )
        # Ey interior (i = 1..Nx-1)
        Ey[:, 1:-1] += dt * (
            -(c**2) * (Bz[:, 1:] - Bz[:, :-1]) / dx
            - Jy_at_Ey[:, 1:-1] / epsilon0
        )

        # ── Step 3: tangential-E boundary condition ─────────────────────────
        if boundary_method in ("mur", "mur+damping"):
            # Mur 1st order on Ex (top/bottom)
            Ex[0,  :] = Ex_prev[1,  :] + alpha_y * (Ex[1,  :] - Ex_prev[0,  :])
            Ex[-1, :] = Ex_prev[-2, :] + alpha_y * (Ex[-2, :] - Ex_prev[-1, :])
            # Mur 1st order on Ey (left/right)
            Ey[:, 0]  = Ey_prev[:, 1]  + alpha_x * (Ey[:, 1]  - Ey_prev[:, 0])
            Ey[:, -1] = Ey_prev[:, -2] + alpha_x * (Ey[:, -2] - Ey_prev[:, -1])
        else:  # "none" or "damping" → PEC walls (tangential E = 0)
            Ex[0,  :] = 0.0
            Ex[-1, :] = 0.0
            Ey[:, 0]  = 0.0
            Ey[:, -1] = 0.0

        if boundary_method in ("damping", "mur+damping"):
            # ⚠ artificially lossy multiplicative mask
            Ex *= mask_ex
            Ey *= mask_ey

        # ── Diagnostics on cell-center grid ─────────────────────────────────
        Ex_c = 0.5 * (Ex[:-1, :] + Ex[1:, :])      # (Ny, Nx)
        Ey_c = 0.5 * (Ey[:, :-1] + Ey[:, 1:])      # (Ny, Nx)

        # Dipole moment (uses ρ on Bz centers)
        px_arr[n], py_arr[n] = compute_dipole_moment(rho_bz, X, Y, dx, dy)

        # Total field energy on each native Yee grid (preferred Yee form)
        energy_hist[n] = (
            0.5 * np.sum(epsilon0 * Ex**2) * dx * dy
            + 0.5 * np.sum(epsilon0 * Ey**2) * dx * dy
            + 0.5 * np.sum(Bz**2 / mu0)     * dx * dy
        )

        # Poynting & ring fluxes on cell centers
        Sx, Sy = compute_poynting(Ex_c, Ey_c, Bz, mu0=mu0)
        _, _, _, _, fs, fo = measure_flux_ring(Sx, Sy, X, Y,
                                                r_obs=r_obs, ntheta=ntheta_ring)
        flux_signed[n] = fs
        flux_out[n]    = fo

        if extra_r_obs is not None:
            _, _, _, _, _, fo2 = measure_flux_ring(Sx, Sy, X, Y,
                                                    r_obs=extra_r_obs,
                                                    ntheta=ntheta_ring)
            flux_r2[n] = fo2
        for r_extra in extra_rings_list:
            _, _, _, _, _, fo_e = measure_flux_ring(Sx, Sy, X, Y,
                                                     r_obs=r_extra,
                                                     ntheta=ntheta_ring)
            flux_extras[float(r_extra)][n] = fo_e

        # Sampled continuity / Gauss diagnostics on cell-center grid
        if n % continuity_interval == 0 and 0 < n < nsteps - 1:
            rho_p, *_ = rotating_rho(X, Y, t-dt, omega, Q, sep, a, b, phase,
                                      dx, dy, **src_kwargs)
            rho_n_, *_ = rotating_rho(X, Y, t+dt, omega, Q, sep, a, b, phase,
                                      dx, dy, **src_kwargs)
            drho_dt = (rho_n_ - rho_p) / (2.0 * dt)
            Jx_c, Jy_c = compute_current(rho_bz, X, Y, omega)
            divJ = divergence2d(Jx_c, Jy_c, dx, dy)
            residual = drho_dt + divJ
            core = (slice(2,-2), slice(2,-2))
            rms_c   = np.sqrt(np.mean(residual[core]**2))
            denom_c = (np.sqrt(np.mean(drho_dt[core]**2))
                       + np.sqrt(np.mean(divJ[core]**2)) + 1e-15)
            cont_rms_list.append(rms_c)
            cont_norm_list.append(rms_c / denom_c)
            divE = ddx(Ex_c, dx) + ddy(Ey_c, dy)
            gauss_r = divE - rho_bz / epsilon0
            gauss_rms_list.append(np.sqrt(np.mean(gauss_r[core]**2)))
            diag_t.append(t)

        # Frame storage
        if store_frames and n % save_interval == 0:
            frames_Bz.append(Bz.copy())
            frames_rho.append(rho_bz.copy())
            frames_Ex.append(Ex_c.copy())
            frames_Ey.append(Ey_c.copy())
            frame_steps.append(n)

        if progress_ticks and ((n+1) % tick_interval == 0 or (n+1) == nsteps):
            print('✓', end='', flush=True)

    elapsed = time.perf_counter() - t0
    if progress_ticks:
        print(f' done ({elapsed:.1f}s)')

    # Final cell-centered fields (for downstream analysis cells)
    Ex_c_final = 0.5 * (Ex[:-1, :] + Ex[1:, :])
    Ey_c_final = 0.5 * (Ey[:, :-1] + Ey[:, 1:])

    frame_steps_arr = np.array(frame_steps, dtype=int)
    return {
        't':              t_arr,
        # Cell-center (collocated view) — used by downstream cells
        'Ex': Ex_c_final, 'Ey': Ey_c_final, 'Bz': Bz,
        # Native staggered Yee arrays — preserved for inspection
        'Ex_yee': Ex, 'Ey_yee': Ey, 'Bz_yee': Bz,
        'X_ex': X_ex, 'Y_ex': Y_ex, 'X_ey': X_ey, 'Y_ey': Y_ey,
        'rho_last':       rho_bz,
        'px': px_arr, 'py': py_arr,
        'flux_signed':    flux_signed,
        'flux_out':       flux_out,
        'flux_r2':        flux_r2,
        'flux_extras':    flux_extras,
        'energy_history': energy_hist,
        'frames_Bz':      np.array(frames_Bz)  if frames_Bz  else np.empty((0,)),
        'frames_rho':     np.array(frames_rho) if frames_rho else np.empty((0,)),
        'frames_Ex':      np.array(frames_Ex)  if frames_Ex  else np.empty((0,)),
        'frames_Ey':      np.array(frames_Ey)  if frames_Ey  else np.empty((0,)),
        'frame_steps':    frame_steps_arr,
        'frame_t':        t_arr[frame_steps_arr] if len(frame_steps_arr) else np.array([]),
        'diag_t':         np.array(diag_t),
        'cont_rms':       np.array(cont_rms_list),
        'cont_norm':      np.array(cont_norm_list),
        'gauss_rms':      np.array(gauss_rms_list),
        'cfl_ratio':      cfl_ratio,
        'cfl_limit':      cfl_lim,
        'cfl_number_2d':  cfl_number_2d,
        'elapsed':        elapsed,
        'r_obs':          r_obs,
        'extra_r_obs':    extra_r_obs,
        'extra_rings':    extra_rings_list,
        'boundary_method': boundary_method,
        'solver_kind':    'yee',
        'source_type':    source_type,
        'n_ring':         n_ring,
        'ring_r':         ring_r,
        'sigma_pt':       sigma_pt,
    }

print("run_fdtd() defined ✓  (Yee staggered grid, leapfrog time stepping; "
      "boundary_method ∈ {'mur','damping','mur+damping','none'})")


### JC: Another change that I implemented to better conserve field behavior is using a YEE style grid. I found this in Chapter 22 of the book where it talks about EM wave propagation and this style of grid is best when there is a calculatio of two fields.

### The next part I should consider is boundary conditions, without boundary conditions the fields would bounce back and be a ginormous mess. After talks with the instructor a dampening mask would be detrimental in calculating how much flux actually is radiated because the boundary absorbs some of it. A better implimentation would be MUR boundary method which has a confined box and removes all fields and flux after a certain region to remove refletions.

## Section 7 — Absorbing Boundary Conditions and Energy Diagnostics

We expose a single switch `boundary_method` controlling how the outer boundary is treated:

| `boundary_method` | Description | Recommended use |
|---|---|---|
| `"mur"` | Mur first-order ABC at all four edges only | **Default for quantitative radiation measurements** |
| `"damping"` | Smooth multiplicative mask near the edges (no Mur) | Comparison only — see warning |
| `"mur+damping"` | Mur ABC + multiplicative damping mask | Legacy v4 behavior |
| `"none"` | Hard-wall (no absorption) | Stability/reflection check |

> ⚠️ **Important warning about the damping mask.**  
> The mask multiplies $E_x, E_y, B_z$ by a factor $<1$ near the boundary every timestep. This is **not** a perfectly matched layer; it acts as an **artificial lossy medium that removes electromagnetic energy from the simulation**. For total-energy or radiated-power budgets the mask biases the result downward. We therefore use `"mur"` for the main production run, and use `"damping"` only as a comparison. The boundary-comparison plot below makes this loss visible.

### Energy diagnostic

We define the (dimensionless) total field energy

$$U(t) = \tfrac{1}{2}\sum_{i,j}\Bigl[\varepsilon_0\bigl(E_x^2+E_y^2\bigr) + B_z^2/\mu_0\Bigr]\,\Delta x\,\Delta y$$

and store `energy_history[n]` at every timestep. Plotting $U(t)$ for `"none"`, `"damping"`, and `"mur"` shows whether damping is artificially draining radiation energy.

The cell below defines all boundary and solver infrastructure.


In [ ]:
# ── Absorbing damping mask ────────────────────────────────────────────────────
def make_absorbing_mask(X, Y, Lx, Ly, width=1.0, strength=0.015):
    """
    Smooth damping mask: 1.0 in the interior, gradually smaller near boundaries.
    This is a simple approximation, NOT a perfect PML.

    ⚠️  WARNING: applying this mask multiplicatively to Ex, Ey, Bz every step
    acts as an *artificial lossy medium* and removes electromagnetic energy
    from the simulation. Use it only for qualitative reflection suppression
    or as a comparison case — NOT for quantitative energy/power budgets.

    Parameters
    ----------
    width    : physical width of the absorbing layer
    strength : max damping per step (small positive, e.g. 0.015)
    """
    dist_left   = X + Lx/2
    dist_right  = Lx/2 - X
    dist_bottom = Y + Ly/2
    dist_top    = Ly/2 - Y

    dist_edge = np.minimum(
        np.minimum(dist_left, dist_right),
        np.minimum(dist_bottom, dist_top)
    )
    s = np.clip(dist_edge / width, 0.0, 1.0)
    # smoothstep profile: 0 at edge → 1 in interior
    smooth = s * s * (3.0 - 2.0 * s)
    mask = 1.0 - strength * (1.0 - smooth)
    return np.clip(mask, 0.0, 1.0)


# ── Mur first-order ABC ───────────────────────────────────────────────────────
def apply_mur_abc(u, u_prev, c, dt, dx, dy):
    """First-order Mur absorbing boundary condition for a 2D scalar field."""
    u_new = u.copy()
    cx = (c*dt - dx) / (c*dt + dx)
    cy = (c*dt - dy) / (c*dt + dy)
    u_new[:, 0]  = u_prev[:, 1]  + cx * (u[:, 1]  - u_prev[:, 0])
    u_new[:, -1] = u_prev[:, -2] + cx * (u[:, -2] - u_prev[:, -1])
    u_new[0, :]  = u_prev[1, :]  + cy * (u[1, :]  - u_prev[0, :])
    u_new[-1, :] = u_prev[-2, :] + cy * (u[-2, :] - u_prev[-1, :])
    return u_new


# ── Boundary-method selector ──────────────────────────────────────────────────
VALID_BOUNDARY_METHODS = ("mur", "damping", "mur+damping", "none")

def apply_boundary(Ex, Ey, Bz, Ex_prev, Ey_prev, Bz_prev, mask,
                   boundary_method, c, dt, dx, dy):
    """Apply the chosen boundary treatment to all three fields."""
    if boundary_method not in VALID_BOUNDARY_METHODS:
        raise ValueError(
            f"boundary_method={boundary_method!r} not in {VALID_BOUNDARY_METHODS}")
    if boundary_method in ("mur", "mur+damping"):
        Ex = apply_mur_abc(Ex, Ex_prev, c, dt, dx, dy)
        Ey = apply_mur_abc(Ey, Ey_prev, c, dt, dx, dy)
        Bz = apply_mur_abc(Bz, Bz_prev, c, dt, dx, dy)
    if boundary_method in ("damping", "mur+damping"):
        # NOTE: the mask is artificially lossy — see WARNING above.
        Ex = Ex * mask
        Ey = Ey * mask
        Bz = Bz * mask
    # boundary_method == "none": hard-wall, no absorption
    return Ex, Ey, Bz


# ── Energy diagnostic ─────────────────────────────────────────────────────────
def compute_field_energy(Ex, Ey, Bz, dx, dy, eps0, mu0):
    """Total electromagnetic energy on the 2D grid (dimensionless units)."""
    return 0.5 * np.sum(eps0 * (Ex**2 + Ey**2) + Bz**2 / mu0) * dx * dy


# ── Poynting vector ───────────────────────────────────────────────────────────
def compute_poynting(Ex, Ey, Bz, mu0=1.0):
    Sx =  Ey * Bz / mu0
    Sy = -Ex * Bz / mu0
    return Sx, Sy


# ── Ring flux measurement ─────────────────────────────────────────────────────
def measure_flux_ring(Sx, Sy, X, Y, r_obs=3.2, ntheta=240):
    """Compute radial Poynting flux around a circle of radius r_obs."""
    theta  = np.linspace(0, 2*np.pi, ntheta, endpoint=False)
    dtheta = 2*np.pi / ntheta
    xr = r_obs * np.cos(theta)
    yr = r_obs * np.sin(theta)

    x_vals = X[0, :]
    y_vals = Y[:, 0]

    ix = np.clip(np.searchsorted(x_vals, xr), 1, len(x_vals)-1)
    iy = np.clip(np.searchsorted(y_vals, yr), 1, len(y_vals)-1)
    ix = np.where(np.abs(x_vals[ix-1] - xr) < np.abs(x_vals[ix] - xr), ix-1, ix)
    iy = np.where(np.abs(y_vals[iy-1] - yr) < np.abs(y_vals[iy] - yr), iy-1, iy)

    Sr       = Sx[iy, ix] * np.cos(theta) + Sy[iy, ix] * np.sin(theta)
    intensity = np.maximum(Sr, 0.0)
    signed_power  = r_obs * np.sum(Sr) * dtheta
    outward_power = r_obs * np.sum(intensity) * dtheta
    return theta, Sr, intensity, dtheta, signed_power, outward_power


# ── Explicit 2D CFL check ─────────────────────────────────────────────────────
CFL_number_main = c * dt_main * np.sqrt(1.0/dx_main**2 + 1.0/dy_main**2)
print("── 2D CFL Check ───────────────────────────────────────")
print(f"  c·Δt·√(1/Δx² + 1/Δy²) = {CFL_number_main:.4f}   (must be ≤ 1)")
if CFL_number_main > 1.0:
    print("  ⚠️  WARNING: CFL number > 1 — solver will be UNSTABLE.")
else:
    print(f"  ✓ stable (margin = {1.0 - CFL_number_main:.4f})")
print("───────────────────────────────────────────────────────")


# ── Visualize absorbing mask ──────────────────────────────────────────────────
abs_mask_main = make_absorbing_mask(X_main, Y_main, Lx_main, Ly_main,
                                     width=1.2, strength=0.018)

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(abs_mask_main, origin='lower', cmap='viridis', vmin=0.97, vmax=1.0,
               extent=[-Lx_main/2, Lx_main/2, -Ly_main/2, Ly_main/2])
plt.colorbar(im, ax=ax, label='Damping factor per step')
ax.set_title('Absorbing damping mask\n(≈ 1 in interior, < 1 near edges)\n'
             '⚠️  artificially lossy — not a PML')
ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout()
plt.show()
print(f"Mask interior min: {abs_mask_main[10:-10,10:-10].min():.6f}, "
      f"edge min: {abs_mask_main.min():.6f}")


### JC: After making the boundary conditions its time to actually run the whole sequence.

## Section 8 — Main Simulation Run



In [ ]:
# ── Main simulation parameters ────────────────────────────────────────────────
# Centralized config for reproducibility — all knobs in one place.
boundary_method_main = "mur"   # "mur" | "damping" | "mur+damping" | "none"

T_main           = 2.0 * np.pi / omega_main
total_time_main  = periods_main * T_main + r_obs_main / c
nsteps_main      = int(np.ceil(total_time_main / dt_main))
save_int_main    = max(1, nsteps_main // 160)   # ~160 frames stored
cont_int_main    = max(1, nsteps_main // 100)   # ~100 diagnostic samples

# Ring-radius sensitivity rings: 0.30 L, 0.40 L, 0.50 L of the *half-width* L=Lx/2
# so they sit progressively closer to the absorbing edge.
L_half           = Lx_main / 2.0
ring_radii_main  = [0.30 * L_half, 0.40 * L_half, 0.50 * L_half]

print("── Main Simulation Parameters ─────────────────────────")
print(f"  boundary_method = {boundary_method_main!r}")
print(f"  Nx, Ny          = {Nx_main}, {Ny_main}")
print(f"  Lx, Ly          = {Lx_main}, {Ly_main}")
print(f"  dx, dy, dt      = {dx_main:.5f}, {dy_main:.5f}, {dt_main:.5f}")
print(f"  omega_main      = {omega_main}")
print(f"  T_main          = {T_main:.4f}")
print(f"  periods_main    = {periods_main}")
print(f"  total_time      = {total_time_main:.4f}")
print(f"  nsteps_main     = {nsteps_main}")
print(f"  CFL (1D ratio)  = {c*dt_main/dx_main:.4f}")
print(f"  CFL (2D number) = {CFL_number_main:.4f}   (must be ≤ 1)")
print(f"  r_obs_main      = {r_obs_main}")
print(f"  ring_radii      = {[f'{r:.2f}' for r in ring_radii_main]}")
print(f"  save_interval   = {save_int_main}")
print(f"  SOURCE_TYPE     = {SOURCE_TYPE}")
if SOURCE_TYPE == 'circular':
    print(f"  N_ring          = {N_ring}")
    print(f"  ring_radius     = {ring_radius}")
    print(f"  sigma_pt        = {sigma_pt}")
print("───────────────────────────────────────────────────────")

# Run — also track a second ring at r=r_obs+0.2 for wave-speed test, plus the
# three sensitivity rings above.
r_wave1 = r_obs_main
r_wave2 = r_obs_main + 0.4

result_main = run_fdtd(
    X_main, Y_main, dx_main, dy_main, dt_main,
    nsteps      = nsteps_main,
    omega       = omega_main,
    Q=Q_main, sep=sep_main, a=a_main, b=b_main, phase=phase_main,
    r_obs       = r_obs_main,
    ntheta_ring = 360,
    save_interval      = save_int_main,
    store_frames       = True,
    continuity_interval= cont_int_main,
    extra_r_obs = r_wave2,                 # for wave-speed test
    extra_rings = ring_radii_main,         # for radius-sensitivity test
    abs_width   = 1.2,
    abs_strength= 0.018,
    boundary_method = boundary_method_main,
    progress_ticks = True,
    label = 'Main run',
    source_type = SOURCE_TYPE,
    n_ring      = N_ring,
    ring_r      = ring_radius,
    sigma_pt    = sigma_pt,
)

# ── Summary statistics ────────────────────────────────────────────────────────
t_start_avg_main = r_obs_main / c + 2.0 * T_main
mask_avg = result_main['t'] >= t_start_avg_main
mean_flux_avg    = np.mean(result_main['flux_out'][mask_avg]) if mask_avg.any() else np.nan
std_flux_avg     = np.std( result_main['flux_out'][mask_avg]) if mask_avg.any() else np.nan
med_cont_norm    = np.nanmedian(result_main['cont_norm'])
med_gauss_rms    = np.nanmedian(result_main['gauss_rms'])

print("\n── Main Run Summary ────────────────────────────────────")
print(f"  Boundary method       : {result_main['boundary_method']}")
print(f"  Elapsed runtime       : {result_main['elapsed']:.1f} s")
print(f"  CFL ratio             : {result_main['cfl_ratio']:.4f}  (limit {result_main['cfl_limit']:.4f})")
print(f"  CFL number (2D)       : {result_main['cfl_number_2d']:.4f}  (must be ≤ 1)")
print(f"  nsteps                : {nsteps_main}")
print(f"  Final field energy U  : {result_main['energy_history'][-1]:.4e}")
print(f"  Mean outward flux     : {mean_flux_avg:.4e}  (averaged after t = {t_start_avg_main:.2f})")
print(f"  Std  outward flux     : {std_flux_avg:.4e}")
print(f"  Median cont norm      : {med_cont_norm:.4e}")
print(f"  Median Gauss RMS      : {med_gauss_rms:.4e}")
print("───────────────────────────────────────────────────────")


## Section 9 — Dipole Moment Validation

The dipole moment $\mathbf{p}(t) = \int \mathbf{r}\,\rho \,dA$ should rotate as a circle in the $(p_x, p_y)$ plane at angular frequency $\omega_\text{main}$. We extract it directly from the tracked charge density.

In [ ]:
px_m = result_main['px']
py_m = result_main['py']
t_m  = result_main['t']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# px, py vs time
axes[0].plot(t_m, px_m, label='$p_x(t)$', alpha=0.85)
axes[0].plot(t_m, py_m, label='$p_y(t)$', alpha=0.85)
axes[0].set_title('Dipole moment components vs time')
axes[0].set_xlabel('t'); axes[0].set_ylabel('p')
axes[0].grid(alpha=0.3); axes[0].legend()

# Phase portrait
axes[1].plot(px_m, py_m, lw=0.8, alpha=0.7)
axes[1].set_aspect('equal')
axes[1].set_title('Phase portrait $p_y$ vs $p_x$')
axes[1].set_xlabel('$p_x$'); axes[1].set_ylabel('$p_y$')
axes[1].grid(alpha=0.3)

# Estimate amplitude and frequency via FFT
from numpy.fft import rfft, rfftfreq
N_fft   = len(px_m)
freqs   = rfftfreq(N_fft, d=dt_main)
pxfft   = np.abs(rfft(px_m))
peak_f  = freqs[np.argmax(pxfft[1:])+1]
amp_px  = np.sqrt(2) * np.std(px_m)   # RMS amplitude

axes[2].plot(freqs[:N_fft//4], pxfft[:N_fft//4])
axes[2].axvline(omega_main/(2*np.pi), color='r', ls='--', lw=1.2,
                label=f'$f = \\omega/2\\pi = {omega_main/(2*np.pi):.3f}$')
axes[2].set_title('FFT of $p_x(t)$')
axes[2].set_xlabel('frequency'); axes[2].set_ylabel('|FFT|')
axes[2].set_xlim(0, omega_main/np.pi)
axes[2].grid(alpha=0.3); axes[2].legend()

plt.suptitle('Dipole Moment Validation', fontsize=13)
plt.tight_layout()
plt.show()

print(f"RMS dipole amplitude: {amp_px:.4f}")
print(f"Dominant FFT frequency: {peak_f:.4f}  (expected {omega_main/(2*np.pi):.4f})")

### JC: This verifies that the charge densities doesnt spread apart or lose any charge. I didnt simulate for large time scales as I only care about the initial wave propagation and then a period after since im looking at the behavior of the fields  and how much flux goes out. Since I know its periodic I can assume it continues this behavior for infinity, So im only limiting it to 3 periods for sake of computation power and time.

## Section 10 — Field Snapshots

We show $B_z(x,y)$ and $|E|(x,y)$ at four timesteps:
1. **Early** — just after source turn-on
2. **Near arrival** — around $t \approx r_\text{obs}/c$
3. **Later radiation** — established wavefronts
4. **Final step** — end of simulation

In [ ]:
n_frames = len(result_main['frames_Bz'])
assert n_frames > 0, "No frames stored. Increase save_interval or re-run with store_frames=True."

# Target fractions: early / near-arrival / later / final
t_arrival = r_obs_main / c
frac_arrival = t_arrival / total_time_main
snap_fracs   = [0.0001, frac_arrival, 0.6, 1.0]
snap_labels  = ['Early (t ≪ r/c)', f'Near arrival (t ≈ {t_arrival:.2f})', 'Later radiation', 'Final step']

snap_idxs = [int(np.clip(round(f * (n_frames - 1)), 0, n_frames-1)) for f in snap_fracs]

fig, axes = plt.subplots(2, len(snap_idxs), figsize=(4.2*len(snap_idxs), 8))
ext = [-Lx_main/2, Lx_main/2, -Ly_main/2, Ly_main/2]

for col, (si, lbl) in enumerate(zip(snap_idxs, snap_labels)):
    t_snap = result_main['frame_t'][si]
    Bz_s   = result_main['frames_Bz'][si]
    Ex_s   = result_main['frames_Ex'][si]
    Ey_s   = result_main['frames_Ey'][si]
    Emag   = np.hypot(Ex_s, Ey_s)

    vlim_B = max(np.abs(Bz_s).max(), 1e-12)
    vlim_E = max(Emag.max(), 1e-12)

    im0 = axes[0, col].imshow(Bz_s, origin='lower', cmap='RdBu',
                               vmin=-vlim_B, vmax=vlim_B, extent=ext)
    overlay_source(axes[0, col], t=t_snap)
    axes[0, col].set_title(f'{lbl}\nt = {t_snap:.2f}')
    axes[0, col].set_xlabel('x'); axes[0, col].set_ylabel('y')
    plt.colorbar(im0, ax=axes[0, col], fraction=0.046, label='$B_z$')

    im1 = axes[1, col].imshow(Emag, origin='lower', cmap='inferno',
                               vmin=0, vmax=vlim_E, extent=ext)
    overlay_source(axes[1, col], t=t_snap)
    axes[1, col].set_title(f'$|E|$  t = {t_snap:.2f}')
    axes[1, col].set_xlabel('x'); axes[1, col].set_ylabel('y')
    plt.colorbar(im1, ax=axes[1, col], fraction=0.046, label='$|E|$')

# One legend for the whole figure
handles, labels = axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc='upper center', ncol=2,
               bbox_to_anchor=(0.5, 1.02), fontsize=9, frameon=False)

plt.suptitle('Field Snapshots: $B_z$ (top row) and $|E|$ (bottom row)\n'
             'Red = + source, Blue = − source, dotted lime = orbit',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


### JC: This follows what I expect, an e field propagating outward radially, and since the current is circular on the x-y plane according to the right hand rule and how a B field is created by Ampere's and Biot Savarts law in EM the b field should propagate in the z direction, in and out depending on what charge the current is. Since there is a positive and negative current by the rotating dipole, I can confirm that there is a propagation in both the positive and negative z axis. 

### Next part is radiation, if there is a magnetic and electric field, you have electromagnetic radiation( IE flux IE energy coming out). How to compute that was stated in griffiths to be, in a vacuum the direction of electromagnetic radiation is defined by the poynting vector.

## Section 11 — Poynting Vector and Energy Flow

The Poynting vector $\mathbf{S} = \mathbf{E} \times \mathbf{B}/\mu_0$ describes electromagnetic energy flux.

In the TMz convention used here:
$$S_x = \frac{E_y B_z}{\mu_0}, \quad S_y = \frac{-E_x B_z}{\mu_0}$$

**Expected behavior:**
- **Near the source**: reactive, circulating patterns  
- **Far from the source**: outward-directed energy transport consistent with radiation

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

Ex_f = result_main['Ex']
Ey_f = result_main['Ey']
Bz_f = result_main['Bz']
Sx_f, Sy_f = compute_poynting(Ex_f, Ey_f, Bz_f, mu0=mu0)

# Time of the final step (last entry of t_arr)
t_final_main = result_main['t'][-1]

# 1-D coordinate arrays (streamplot expects strictly-increasing 1-D x and y)
x_main_1d = X_main[0, :]
y_main_1d = Y_main[:, 0]

ext = [-Lx_main/2, Lx_main/2, -Ly_main/2, Ly_main/2]

# ── Signed scalars (full ± range) ────────────────────────────────────────────
# Radial Poynting  Sr = (S · r̂)  →  positive = outward energy flow
# Radial electric  Er = (E · r̂)  →  positive = outward-pointing field
R_main = np.hypot(X_main, Y_main)
R_safe = np.where(R_main > 0, R_main, 1.0)
xhat   = X_main / R_safe
yhat   = Y_main / R_safe
Sr_f   = Sx_f * xhat + Sy_f * yhat
Er_f   = Ex_f * xhat + Ey_f * yhat

vlim_Sr = max(np.abs(Sr_f).max(), 1e-12)
vlim_Er = max(np.abs(Er_f).max(), 1e-12)
vlim_B  = max(np.abs(Bz_f).max(), 1e-12)

# ── Custom diverging colormaps with BLACK at zero (high contrast) ───────────
# Poynting:  cyan (inward)  → black → hot-orange (outward)
poynting_cmap = LinearSegmentedColormap.from_list(
    'poynting_black',
    ["#ae00ff", '#0080ff', '#000000', "#d4ff00", '#ffff00'],
    N=256,
)
# Electric:  magenta (inward) → black → bright green (outward)
efield_cmap = LinearSegmentedColormap.from_list(
    'efield_black',
    ["#ff0080", '#8000ff', '#000000', "#09ff00", "#eeff00"],
    N=256,
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.4))

# ── Left: Poynting flow — black-background diverging cmap ───────────────────
axes[0].set_facecolor('black')
im0 = axes[0].imshow(Sr_f, origin='lower', cmap=poynting_cmap,
                     vmin=-vlim_Sr, vmax=vlim_Sr, extent=ext)
plt.colorbar(im0, ax=axes[0], fraction=0.046,
             label=r'$S_r = \mathbf{S}\cdot\hat{\mathbf{r}}$')
axes[0].streamplot(
    x_main_1d, y_main_1d, Sx_f, Sy_f,
    color='white', density=1.6, linewidth=0.8, arrowsize=1.1,
)
overlay_source(axes[0], t=t_final_main)
axes[0].set_xlim(ext[0], ext[1]); axes[0].set_ylim(ext[2], ext[3])
axes[0].set_aspect('equal')
axes[0].set_title('Poynting flow  (orange = outward, cyan = inward)')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

# ── Middle: Electric flow — black-background diverging cmap ─────────────────
axes[1].set_facecolor('black')
im1 = axes[1].imshow(Er_f, origin='lower', cmap=efield_cmap,
                     vmin=-vlim_Er, vmax=vlim_Er, extent=ext)
plt.colorbar(im1, ax=axes[1], fraction=0.046,
             label=r'$E_r = \mathbf{E}\cdot\hat{\mathbf{r}}$')
axes[1].streamplot(
    x_main_1d, y_main_1d, Ex_f, Ey_f,
    color='white', density=1.6, linewidth=0.8, arrowsize=1.1,
)
circle1 = plt.Circle((0, 0), r_obs_main, fill=False, color='lime', lw=1.2, ls='--',
                     label=f'$r_{{obs}}={r_obs_main}$')
axes[1].add_patch(circle1)
overlay_source(axes[1], t=t_final_main)
axes[1].legend(loc='upper right', fontsize=8)
axes[1].set_xlim(ext[0], ext[1]); axes[1].set_ylim(ext[2], ext[3])
axes[1].set_aspect('equal')
axes[1].set_title('Electric flow  (green = outward, magenta = inward)')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')

# ── Right: Bz field — distinct white-centered diverging cmap ────────────────
im2 = axes[2].imshow(Bz_f, origin='lower', cmap='RdBu',
                     vmin=-vlim_B, vmax=vlim_B, extent=ext)
plt.colorbar(im2, ax=axes[2], fraction=0.046, label='$B_z$')
axes[2].streamplot(
    x_main_1d, y_main_1d, Sx_f, Sy_f,
    color='black', density=1.6, linewidth=0.7, arrowsize=1.0,
)
circle2 = plt.Circle((0, 0), r_obs_main, fill=False, color='lime', lw=1.2, ls='--',
                     label=f'$r_{{obs}}={r_obs_main}$')
axes[2].add_patch(circle2)
overlay_source(axes[2], t=t_final_main)
axes[2].legend(loc='upper right', fontsize=8)
axes[2].set_xlim(ext[0], ext[1]); axes[2].set_ylim(ext[2], ext[3])
axes[2].set_aspect('equal')
axes[2].set_title('$B_z$ field  with Poynting streamlines')
axes[2].set_xlabel('x'); axes[2].set_ylabel('y')

plt.suptitle(f'Poynting, Electric & $B_z$ Fields at Final Step (t = {t_final_main:.2f})\n'
             'Red = + source, Blue = − source, dotted lime = orbit',
             fontsize=12)
plt.tight_layout()
plt.show()


### JC: For practical reasons and also to better quantify the field propagation and shape, I decided to use np.streamplot. This helps with validating through the right hand rule, is the poynting vector actually getting the cross product. I validated that yes it did regardless of the dipole shape. 

### Additionally this creates a much better visualization of the dipole like behavior of the e field compared to the magnitude plot. Depending on the size of the box in relation to the separation and size of the dipole, we can examine far field vs near field effects. Near field effects include the circulation fo both poynting vector and E field. However in far field we can see that the poynting vector actually points radially outward when the dipole losees its curvature because of the wave propagation.



In [ ]:
t_arr      = result_main['t']
flux_out   = result_main['flux_out']
flux_sign  = result_main['flux_signed']

t_prop     = r_obs_main / c
t_avg_start = t_prop + 2.0 * T_main
mask_avg   = t_arr >= t_avg_start

mean_f = np.mean(flux_out[mask_avg]) if mask_avg.any() else np.nan
std_f  = np.std( flux_out[mask_avg]) if mask_avg.any() else np.nan

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(t_arr, flux_out,  label='Outward flux', lw=1.2)
ax.plot(t_arr, flux_sign, label='Signed flux',  lw=0.8, alpha=0.6)
ax.axvline(t_prop,      color='k',      ls='--', lw=1.2, label=f'$r_{{obs}}/c = {t_prop:.2f}$')
ax.axvline(t_avg_start, color='orange', ls=':',  lw=1.2, label=f'Avg start = {t_avg_start:.2f}')
if mask_avg.any():
    ax.axhspan(mean_f - 
    std_f, mean_f + std_f, alpha=0.15, color='tab:green', label='±1σ avg region')
    ax.axhline(mean_f, color='tab:green', ls='-', lw=1.0, label=f'Mean = {mean_f:.3e}')
ax.set_title(f'Ring flux vs time  ($r_{{obs}} = {r_obs_main}$, $\\omega = {omega_main}$)')
ax.set_xlabel('t (dimensionless)'); ax.set_ylabel('Ring flux (dimensionless)')
ax.grid(alpha=0.3); ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Mean outward flux after t = {t_avg_start:.2f}: {mean_f:.4e} ± {std_f:.4e}")

### This helps to quantitavely confirm that the flux stabilizes to a steady average and oscillates back and forth as the E field goes through waves of cancellation. The peak is when two wave fronts colided, being the initial wave that splits as the particle moves.

## Section 12a — Energy Diagnostic and Boundary-Method Comparison

We re-run the **same source** with three boundary methods on a smaller, faster grid:

- `boundary_method="none"` — hard wall (reflections expected)
- `boundary_method="damping"` — multiplicative damping mask only
- `boundary_method="mur"`     — Mur first-order ABC only

For each run we record `energy_history` and the outward ring flux at $r = r_\text{obs}$. The plot makes the **artificial energy loss caused by the damping mask** directly visible.


In [ ]:
# ── Boundary-method comparison run (smaller grid, identical source) ──────────
N_cmp        = 600                        # smaller grid for fast triple-run
L_cmp        = Lx_main                    # same physical size as main run
dx_cmp       = L_cmp / N_cmp
dt_cmp       = cfl_safety * dx_cmp / c
x_cmp        = np.linspace(-L_cmp/2, L_cmp/2, N_cmp, endpoint=False)
y_cmp        = np.linspace(-L_cmp/2, L_cmp/2, N_cmp, endpoint=False)
X_cmp, Y_cmp = np.meshgrid(x_cmp, y_cmp, indexing='xy')

periods_cmp  = max(3.0, periods_main * 0.5)
total_t_cmp  = periods_cmp * T_main + r_obs_main / c
nsteps_cmp   = int(np.ceil(total_t_cmp / dt_cmp))

L_half_cmp   = L_cmp / 2.0
ring_radii_cmp = [0.30 * L_half_cmp, 0.40 * L_half_cmp, 0.50 * L_half_cmp]

boundary_methods_cmp = ["none", "damping", "mur"]
results_cmp = {}

print(f"Boundary comparison: N={N_cmp}, nsteps={nsteps_cmp}, "
      f"omega={omega_main}, periods={periods_cmp}")
for bm in boundary_methods_cmp:
    results_cmp[bm] = run_fdtd(
        X_cmp, Y_cmp, dx_cmp, dx_cmp, dt_cmp,
        nsteps      = nsteps_cmp,
        omega       = 8 * omega_main,
        Q=Q_main, sep=sep_main, a=a_main, b=b_main, phase=phase_main,
        r_obs       = r_obs_main, ntheta_ring=240,
        save_interval = max(1, nsteps_cmp // 20),
        store_frames  = True,
        continuity_interval = max(1, nsteps_cmp // 30),
        extra_rings = ring_radii_cmp,
        abs_width=1.2, abs_strength=0.018,
        boundary_method = bm,
        progress_ticks  = True, label=f'cmp({bm})',
        source_type = SOURCE_TYPE, n_ring=N_ring,
        ring_r=ring_radius, sigma_pt=sigma_pt,
    )

# ── Plot: energy vs time, flux vs time, final Bz snapshots ────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
colors = {'none': 'tab:red', 'damping': 'tab:orange', 'mur': 'tab:blue'}

# Row 0, col 0: energy vs time
ax = axes[0, 0]
for bm in boundary_methods_cmp:
    r = results_cmp[bm]
    ax.plot(r['t'], r['energy_history'], color=colors[bm], lw=1.3,
            label=f'{bm}')
ax.axvline(r_obs_main/c, color='k', ls=':', lw=0.8, label=r'$r_{obs}/c$')
ax.set_xlabel('t'); ax.set_ylabel('Total field energy U(t)')
ax.set_title(f'Energy vs time  (ω={omega_main}, N={N_cmp})')
ax.grid(alpha=0.3); ax.legend(fontsize=9)

# Row 0, col 1: outward flux vs time
ax = axes[0, 1]
for bm in boundary_methods_cmp:
    r = results_cmp[bm]
    ax.plot(r['t'], r['flux_out'], color=colors[bm], lw=1.0, alpha=0.85,
            label=f'{bm}')
ax.axvline(r_obs_main/c, color='k', ls=':', lw=0.8)
ax.set_xlabel('t'); ax.set_ylabel(f'Outward flux at r={r_obs_main}')
ax.set_title('Outward ring flux vs time')
ax.grid(alpha=0.3); ax.legend(fontsize=9)

# Row 0, col 2: late-time mean flux per method (bar)
ax = axes[0, 2]
t_avg_cmp = r_obs_main/c + 1.5 * T_main
means = []
for bm in boundary_methods_cmp:
    r = results_cmp[bm]
    m = r['t'] >= t_avg_cmp
    means.append(np.mean(r['flux_out'][m]) if m.any() else np.nan)
ax.bar(boundary_methods_cmp, means, color=[colors[b] for b in boundary_methods_cmp])
ax.set_ylabel('Mean outward flux (late-time)')
ax.set_title(f'Late-time mean flux (after t={t_avg_cmp:.2f})')
ax.grid(alpha=0.3, axis='y')

# Row 1: final Bz snapshots
ext_cmp = [-L_cmp/2, L_cmp/2, -L_cmp/2, L_cmp/2]
vmax_cmp = max(np.abs(results_cmp[bm]['Bz']).max() for bm in boundary_methods_cmp)
for j, bm in enumerate(boundary_methods_cmp):
    Bz_f = results_cmp[bm]['Bz']
    im = axes[1, j].imshow(Bz_f, origin='lower', cmap='RdBu',
                            vmin=-vmax_cmp, vmax=vmax_cmp, extent=ext_cmp)
    axes[1, j].set_title(f"Final $B_z$ — boundary={bm}\n"
                         f"ω={omega_main}, N={N_cmp}")
    axes[1, j].set_xlabel('x'); axes[1, j].set_ylabel('y')
    plt.colorbar(im, ax=axes[1, j], fraction=0.046, label='$B_z$')

plt.suptitle('Boundary-method comparison: energy, flux, reflections',
             fontsize=13)
plt.tight_layout()
plt.show()

print("\n── Boundary-method summary ─────────────────────────────")
for bm, mflux in zip(boundary_methods_cmp, means):
    Uf = results_cmp[bm]['energy_history'][-1]
    Umax = results_cmp[bm]['energy_history'].max()
    print(f"  {bm:>12s}:  U_final={Uf:.3e}  U_max={Umax:.3e}  "
          f"mean_flux_late={mflux:.3e}")
print("───────────────────────────────────────────────────────")
print("Interpretation:")
print("  • 'none'    — energy keeps growing; reflections contaminate flux.")
print("  • 'damping' — energy is artificially drained: U(t) sags below 'mur'")
print("                even before the wave hits the wall, and ring flux is")
print("                biased downward. This confirms the mask acts as an")
print("                artificial lossy medium.")
print("  • 'mur'     — energy plateaus once the radiated wave starts leaving;")
print("                this is the appropriate boundary for radiation budgets.")
print()
print("CONCLUSION: The damping mask acts as an artificial lossy medium and")
print("removes electromagnetic energy. Mur ABC is more appropriate for")
print("quantitative radiation measurements.")


### JC: a comparison between the boundary conditions and which one is the most realistic, None gives a reflective goopy mess. Damping absorbs the fields as stated before so it isnt reliable for a future Quantitave simulation. Mur still is deemed the best

## Section 12b — Ring-Radius Sensitivity Test

We measure the time-averaged outward Poynting flux on three concentric rings at $r = 0.30\,L_{1/2}$, $0.40\,L_{1/2}$, $0.50\,L_{1/2}$ (with $L_{1/2}=L_x/2$), using the main-run results that already track these rings via `extra_rings`.

**Interpretation:**
- In a clean far-field region inside the physical (non-damped) interior, the measured average power should be **roughly stable with radius**.
- A strong drop with increasing $r$ indicates that either the damping mask or numerical loss is contaminating the result (typically the larger ring has entered the absorbing layer).

We also overlay the same measurement from the `boundary="damping"` comparison run to make the contamination explicit.


In [ ]:
# ── Ring-radius sensitivity ───────────────────────────────────────────────────
def average_extra_rings(result, t_start):
    """Return (radii, mean_outward_flux) over t >= t_start, sorted by radius."""
    t = result['t']
    m = t >= t_start
    radii = sorted(result['flux_extras'].keys())
    means = []
    for r in radii:
        flux_r = result['flux_extras'][r]
        means.append(np.mean(flux_r[m]) if m.any() else np.nan)
    return np.array(radii), np.array(means)

t_avg_main_late = r_obs_main / c + 2.0 * T_main
radii_m, means_main_m = average_extra_rings(result_main, t_avg_main_late)

# Same measurement from the boundary-comparison runs
t_avg_cmp_late = r_obs_main / c + 1.5 * T_main
radii_d, means_d = average_extra_rings(results_cmp['damping'], t_avg_cmp_late)
radii_n, means_n = average_extra_rings(results_cmp['none'],    t_avg_cmp_late)
radii_M, means_M = average_extra_rings(results_cmp['mur'],     t_avg_cmp_late)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(radii_m, means_main_m, 'o-', lw=1.6, color='tab:green',
        label=f'main run ({result_main["boundary_method"]}, N={Nx_main})')
ax.plot(radii_M, means_M, 's--', lw=1.2, color='tab:blue',  label=f'cmp: mur (N={N_cmp})')
ax.plot(radii_d, means_d, 's--', lw=1.2, color='tab:orange',label=f'cmp: damping (N={N_cmp})')
ax.plot(radii_n, means_n, 's--', lw=1.2, color='tab:red',   label=f'cmp: none (N={N_cmp})')
ax.set_xlabel('Ring radius r')
ax.set_ylabel('Mean outward Poynting flux (late-time)')
ax.set_title(f'Ring-radius sensitivity  (ω={omega_main}, '
             f'L={Lx_main}, source={SOURCE_TYPE})')
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("Ring radii (main run):", [f'{r:.3f}' for r in radii_m])
print("Mean flux  (main run):", [f'{v:.3e}' for v in means_main_m])

### JC: By eye you. cant se the drop off in the flux, but numerically it does show numerically. 

## Section 13 — Angular Radiation Pattern

We time-average the outward Poynting intensity $I(\theta)$ over the steady-state portion of the run, and compare to a simple **idealized dipole-like reference** curve.

> **Important caveat:** This is a 2D simulation, not 3D. The exact 3D electric dipole pattern $\propto \sin^2\theta$ does not apply here. Deviations are expected due to: 2D geometry, finite source size, finite observation radius, boundary reflections, and the collocated grid. The reference curve is provided for orientation only.

In [ ]:
ntheta_ang  = 360
theta_ang   = np.linspace(0, 2*np.pi, ntheta_ang, endpoint=False)

# Collect intensity profiles from frames in the averaging window
mask_frame_avg = result_main['frame_t'] >= t_avg_start
frame_idxs_avg = np.where(mask_frame_avg)[0]

if len(frame_idxs_avg) == 0:
    # Fallback: use last quarter of frames
    frame_idxs_avg = np.arange(len(result_main['frame_t']) * 3 // 4,
                                len(result_main['frame_t']))

intensity_sum = np.zeros(ntheta_ang)
for fi in frame_idxs_avg:
    Ex_fi = result_main['frames_Ex'][fi]
    Ey_fi = result_main['frames_Ey'][fi]
    Bz_fi = result_main['frames_Bz'][fi]
    Sx_fi, Sy_fi = compute_poynting(Ex_fi, Ey_fi, Bz_fi, mu0=mu0)
    _, _, intens_i, _, _, _ = measure_flux_ring(Sx_fi, Sy_fi, X_main, Y_main,
                                                 r_obs=r_obs_main, ntheta=ntheta_ang)
    intensity_sum += intens_i

intensity_avg = intensity_sum / max(len(frame_idxs_avg), 1)
i_max = intensity_avg.max()
if i_max < 1e-15:
    i_max = 1.0
intensity_norm = intensity_avg / i_max

# Reference: idealized dipole-like reference (not exact 3D sin²θ)
# For a 2D rotating dipole, a uniform "donut" or cos²-based reference is approximate.
# We use a simple constant + cos²(2θ) reference, labeled honestly.
ref_curve  = 0.5 + 0.5 * np.cos(2 * theta_ang)**2
ref_norm   = ref_curve / ref_curve.max()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Polar plot
ax_pol = plt.subplot(121, projection='polar')
ax_pol.plot(theta_ang, intensity_norm, lw=1.5, label='Simulation (normalized)')
ax_pol.plot(theta_ang, ref_norm, lw=1.2, ls='--', color='tab:orange',
            label='Idealized dipole-like reference')
ax_pol.set_title('Angular intensity pattern\n(time-averaged)')
ax_pol.legend(loc='lower right', fontsize=8)

# Cartesian
axes[1].plot(np.degrees(theta_ang), intensity_norm, lw=1.5, label='Simulation')
axes[1].plot(np.degrees(theta_ang), ref_norm, lw=1.2, ls='--', color='tab:orange',
             label='Idealized dipole-like reference\n(NOT exact 3D sin²θ)')
axes[1].set_xlabel('$\\theta$ (degrees)'); axes[1].set_ylabel('Normalized intensity')
axes[1].set_title('Angular intensity (Cartesian view)')
axes[1].grid(alpha=0.3); axes[1].legend(fontsize=9)

plt.suptitle('Angular Radiation Pattern (deviations from reference are expected — see caption)', fontsize=11)
plt.tight_layout()
plt.show()

print(f"Frames averaged: {len(frame_idxs_avg)}  (from t = {result_main['frame_t'][frame_idxs_avg[0]]:.2f})")

### JC: even though this isnt an ideal dipole it still goes thrugh behavior similar to one. Angular intensity is periodic and there are angles at which there is more flux than others.

## Section 14 — Wave Speed Validation

We measure the propagation speed of the wavefront by tracking when the outward ring flux first rises above a threshold at two radii:
- $r_1 = r_\text{obs,main} = 3.2$ (from the main run)
- $r_2 = 3.5$ (tracked simultaneously via `extra_r_obs`)

**Method:** find $t_\text{arrival}$ as the first time the signal exceeds 10% of its maximum. Then:

$$c_\text{measured} = \frac{r_2 - r_1}{t_\text{arrival,2} - t_\text{arrival,1}}$$

If the result deviates from $c = 1$, this reflects: numerical dispersion on a collocated grid, boundary effects, or threshold-crossing noise.

In [ ]:
sig1 = result_main['flux_out']                   # ring at r_obs_main = 3.2
sig2 = result_main['flux_r2']                    #ring at r_wave2    = 2.2
t_ws = result_main['t']

def find_arrival(sig, t_arr, threshold_frac=0.001, t_min=0.0):
    """First time signal exceeds threshold_frac * max(signal) after t_min."""
    mask = t_arr >= t_min
    if not mask.any():
        return np.nan
    sig_m = sig[mask]
    t_m   = t_arr[mask]
    smax  = sig_m.max()
    if smax < 1e-15:
        return np.nan
    above = np.where(sig_m >= threshold_frac * smax)[0]
    return float(t_m[above[0]]) if len(above) else np.nan

t1 = find_arrival(sig1, t_ws, threshold_frac=0.10)
t2 = find_arrival(sig2, t_ws, threshold_frac=0.10)

if sig2 is None or not np.any(sig2 > 0):
    print("WARNING: extra_r_obs ring flux not available. Re-run with extra_r_obs set.")
    c_measured = np.nan
else:
    dt_wave    = t2 - t1
    dr_wave    = r_wave2 - r_obs_main
    c_measured = dr_wave / dt_wave if dt_wave > 0 else np.nan

print(f"r1 = {r_obs_main},   arrival t1 = {t1:.16f}")
print(f"r2 = {r_wave2},   arrival t2 = {t2:.16f}")
print(f"Δr = {r_wave2 - r_obs_main:.2f},   Δt = {t2 - t1:.16f}")
print(f"c_measured = {c_measured:.16f}  (expected c = {c:.1f})")
if not np.isnan(c_measured):
    print(f"Relative error: {abs(c_measured - c)/c * 100:.1f}%")

# Plot
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(t_ws, sig1, label=f'Outward flux at $r_1={r_obs_main}$', lw=1.2)
if sig2 is not None:
    ax.plot(t_ws, sig2, label=f'Outward flux at $r_2={r_wave2}$', lw=1.2)
if not np.isnan(t1):
    ax.axvline(t1, color='tab:blue', ls='--', lw=1.0, label=f'$t_1 = {t1:.3f}$')
if not np.isnan(t2):
    ax.axvline(t2, color='tab:orange', ls='--', lw=1.0, label=f'$t_2 = {t2:.3f}$')
ax.axvline(r_obs_main/c, color='gray', ls=':', lw=0.9, label=f'$r_1/c = {r_obs_main/c:.2f}$')
ax.set_xlabel('t'); ax.set_ylabel('Outward ring flux')
ax.set_title(f'Wave speed validation: $c_{{meas}} = {c_measured}$  (expected {c:.1f})')
ax.grid(alpha=0.3); ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.show()

### JC: This recent run I went for speed to get some plots and figures for the report and presentation, but this was a great quantitative measure if the waves are propagating correctly. As the flux (light) should propagate at the same time as the fields, which is the speed of light. The lowest ive ever got this value was 0.09% but that was at a large radius witha large grid and small timestep

## Section 15 — Continuity and Gauss-Law Diagnostics

These residuals are computed at sampled timesteps during the main simulation.

- **Continuity residual:** checks $\partial_t\rho + \nabla\cdot\mathbf{J} \approx 0$. A normalized value $\ll 1$ confirms the prescribed source is locally consistent with charge conservation.  
- **Gauss-law residual:** checks $\nabla\cdot\mathbf{E} \approx \rho/\varepsilon_0$. Because $\mathbf{E}$ is not initialized from a Poisson solve and the solver does not enforce this constraint, a nonzero residual is expected. It is a **numerical diagnostic**, not a measure of solver correctness.

In [ ]:
diag_t_m  = result_main['diag_t']
cont_norm = result_main['cont_norm']
gauss_rms = result_main['gauss_rms']

if len(diag_t_m) == 0:
    print("No diagnostic samples found. Increase nsteps or decrease continuity_interval.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].semilogy(diag_t_m, cont_norm, color='tab:green', lw=1.2)
    axes[0].axhline(np.nanmedian(cont_norm), color='tab:green', ls='--', lw=1.0,
                    label=f'Median = {np.nanmedian(cont_norm):.2e}')
    axes[0].set_title('Normalized continuity residual\n$\\|\\partial_t\\rho + \\nabla\\cdot J\\|_{RMS} / (\\|\\partial_t\\rho\\| + \\|\\nabla\\cdot J\\|)$')
    axes[0].set_xlabel('t'); axes[0].set_ylabel('Normalized residual')
    axes[0].grid(alpha=0.3, which='both'); axes[0].legend()

    axes[1].semilogy(diag_t_m, gauss_rms, color='tab:purple', lw=1.2)
    axes[1].axhline(np.nanmedian(gauss_rms), color='tab:purple', ls='--', lw=1.0,
                    label=f'Median = {np.nanmedian(gauss_rms):.2e}')
    axes[1].set_title('Gauss-law residual RMS\n$\\|\\nabla\\cdot E - \\rho/\\varepsilon_0\\|_{RMS}$\n(not expected to be zero in this solver)')
    axes[1].set_xlabel('t'); axes[1].set_ylabel('RMS residual')
    axes[1].grid(alpha=0.3, which='both'); axes[1].legend()

    plt.tight_layout()
    plt.show()

    print(f"Median continuity norm : {np.nanmedian(cont_norm):.4e}")
    print(f"Median Gauss RMS       : {np.nanmedian(gauss_rms):.4e}")

### JC: Continuity and Gauss law residual comes from Griffiths theory of charge and field conservation. It makes sense that the normalized continuity residual of current is a semi small value considering both charges are a value of 10 each. But id have to do more research why gauss law isnt consserved. Could be because we arent calculating the electric fields in teh z direction too.

## Section 16 — Resolution Convergence Study

We run four short simulations at increasing grid resolution to check how the mean outward flux and diagnostic residuals depend on grid spacing $\Delta x$.

A well-behaved solver should show improving (decreasing) residuals and converging flux estimates as resolution increases.

In [ ]:
RUN_CONVERGENCE = True   # set False to skip (shows empty table)

conv_resolutions = np.linspace(100, 900, 8)
conv_periods     = 3.0    # shorter run for speed
conv_omega       = omega_main
conv_r_obs       = r_obs_main
conv_T           = 2.0 * np.pi / conv_omega
conv_rows        = []

if RUN_CONVERGENCE:
    print("Running resolution convergence study…")
    for Nconv in conv_resolutions:
        Nconv = int(Nconv)
        dx_c  = 10.0 / Nconv
        dy_c  = dx_c
        dt_c  = cfl_safety * dx_c / c
        x_c   = np.linspace(-5.0, 5.0, Nconv, endpoint=False)
        y_c   = np.linspace(-5.0, 5.0, Nconv, endpoint=False)
        Xc, Yc = np.meshgrid(x_c, y_c, indexing='xy')

        total_t_c = conv_periods * conv_T + conv_r_obs / c
        nsteps_c  = int(np.ceil(total_t_c / dt_c))
        t_avg_c   = conv_r_obs / c + 1.5 * conv_T

        rc = run_fdtd(
            Xc, Yc, dx_c, dy_c, dt_c,
            nsteps=nsteps_c, omega=conv_omega,
            Q=Q_main, sep=sep_main, a=a_main, b=b_main, phase=phase_main,
            r_obs=conv_r_obs, ntheta_ring=240,
            save_interval=max(1, nsteps_c//40), store_frames=False,
            continuity_interval=max(1, nsteps_c//60),
            abs_width=1.2, abs_strength=0.018,
            progress_ticks=True, label=f'Conv N={Nconv}',
        )
        mask_c = rc['t'] >= t_avg_c
        conv_rows.append({
            'N':             Nconv,
            'dx':            dx_c,
            'dt':            dt_c,
            'nsteps':        nsteps_c,
            'mean_flux':     float(np.mean(rc['flux_out'][mask_c])) if mask_c.any() else np.nan,
            'med_cont_norm': float(np.nanmedian(rc['cont_norm'])),
            'med_gauss_rms': float(np.nanmedian(rc['gauss_rms'])),
        })

    conv_df = pd.DataFrame(conv_rows)
    print("\nResolution convergence table:")
    display(conv_df)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    axes[0].plot(conv_df['dx'], conv_df['mean_flux'], 'o-')
    axes[0].set_xlabel('$\\Delta x$'); axes[0].set_ylabel('Mean outward flux')
    axes[0].set_title('Mean flux vs resolution'); axes[0].grid(alpha=0.3)

    axes[1].semilogy(conv_df['dx'], conv_df['med_cont_norm'], 's-', color='tab:green')
    axes[1].set_xlabel('$\\Delta x$'); axes[1].set_ylabel('Median continuity norm')
    axes[1].set_title('Continuity residual vs resolution'); axes[1].grid(alpha=0.3, which='both')

    axes[2].semilogy(conv_df['dx'], conv_df['med_gauss_rms'], '^-', color='tab:purple')
    axes[2].set_xlabel('$\\Delta x$'); axes[2].set_ylabel('Median Gauss RMS')
    axes[2].set_title('Gauss residual vs resolution'); axes[2].grid(alpha=0.3, which='both')

    plt.suptitle('Resolution Convergence Study', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    conv_df = pd.DataFrame(columns=['N','dx','dt','nsteps','mean_flux','med_cont_norm','med_gauss_rms'])
    print("RUN_CONVERGENCE = False — skipped. Set True to run the convergence study.")

### JC: As usual always do a resolution convergence test to see if the values converge with a given value of dx. This shows that it starts to plateau, but still curiously as it plateaus in the continuity and mean flux, its inverse with the gauss residual.

## Section 17 — Frequency Scaling Study

For an ideal 3D electric dipole, Larmor's formula gives $P \propto \omega^4 p_0^2$.

> **Important:** This 2D finite-source simulation is **not** expected to exactly reproduce the $\omega^4$ scaling. Deviations are expected due to 2D geometry, finite source size, finite grid, and boundary effects. However, a **positive power-law trend** with frequency is a useful physical validation.

We fit $\log P = \alpha \log\omega + \text{const}$ and compare $\alpha$ to the ideal value of 4.

In [ ]:
# ── Frequency sweep grid (coarser for speed) ─────────────────────────────────
Nx_freq = Ny_freq = 600
Lx_freq = Ly_freq = 2.0
dx_freq = Lx_freq / Nx_freq
dt_freq = cfl_safety * dx_freq / c
x_freq  = np.linspace(-Lx_freq/2, Lx_freq/2, Nx_freq, endpoint=False)
y_freq  = np.linspace(-Ly_freq/2, Ly_freq/2, Ny_freq, endpoint=False)
X_freq, Y_freq = np.meshgrid(x_freq, y_freq, indexing='xy')

omegas_sweep = np.array([0.4, 0.6, 0.8, 1.0, 1.5, 2.0, 3.0, 4.0])  # focus on lower frequencies for clearer scaling trends
#omegas_sweep = np.array([ 2.0, 2.5 , 3.0, 3.5, 4.0, 5.0, 6.0, 7.0, 8.0])  # focus on higher frequencies for clearer scaling trends
freq_periods = 8.0      # periods per run
r_obs_freq   = 0.9 * (Lx_freq / 2.0)   # smaller radius for speed (but still in far-field)

freq_rows = []
print("Running frequency scaling study…")
for om in omegas_sweep:
    T_om     = 2.0 * np.pi / om
    total_t  = freq_periods * T_om + r_obs_freq / c
    nsteps_f = int(np.ceil(total_t / dt_freq))
    t_avg_f  = r_obs_freq / c + 1.5 * T_om

    rf = run_fdtd(
        X_freq, Y_freq, dx_freq, dx_freq, dt_freq,
        nsteps=nsteps_f, omega=float(om),
        Q=Q_main, sep=sep_main, a=a_main, b=b_main, phase=phase_main,
        r_obs=r_obs_freq, ntheta_ring=240,
        save_interval=max(1, nsteps_f//40), store_frames=False,
        continuity_interval=max(1, nsteps_f//50),
        abs_width=1.2, abs_strength=0.018,
        progress_ticks=True, label=f'ω={om:.1f}',
    )
    mask_f  = rf['t'] >= t_avg_f
    p0_est  = float(np.sqrt(np.mean(rf['px'][mask_f]**2 + rf['py'][mask_f]**2))) if mask_f.any() else np.nan
    avg_out = float(np.mean(rf['flux_out'][mask_f])) if mask_f.any() else np.nan

    freq_rows.append({
        'omega': float(om),
        'T':     float(T_om),
        'nsteps': nsteps_f,
        'p0_est': p0_est,
        'avg_flux_out': avg_out,
        'med_cont_norm': float(np.nanmedian(rf['cont_norm'])),
        'med_gauss_rms': float(np.nanmedian(rf['gauss_rms'])),
    })

freq_df = pd.DataFrame(freq_rows)
print("\nFrequency scaling table:")
display(freq_df)

# ── Power-law fit ──────────────────────────────────────────────────────────────
omegas_fit  = freq_df['omega'].values
P_out       = freq_df['avg_flux_out'].values
p0_arr      = freq_df['p0_est'].values

valid = P_out > 0
if valid.sum() >= 2:
    log_om  = np.log(omegas_fit[valid])
    log_P   = np.log(P_out[valid])
    coeffs  = np.polyfit(log_om, log_P, 1)
    alpha_fit = coeffs[0]
    print(f"\nFitted log-log slope α = {alpha_fit:.3f}  (ideal 3D dipole: α = 4)")
else:
    alpha_fit = np.nan
    print("Not enough valid points for power-law fit.")

# Reference ω^4 p0² trend (normalized)
ref_omega4  = omegas_fit**4 * p0_arr**2
ref_omega4  = ref_omega4 / np.nanmax(ref_omega4)
P_norm      = P_out / np.nanmax(P_out)

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.loglog(omegas_fit, P_norm,     'o-',  lw=1.5, label=f'Simulation (normalized), α ≈ {alpha_fit:.2f}')
ax.loglog(omegas_fit, ref_omega4, 's--', lw=1.2, color='tab:orange',
          label='$\\omega^4 p_0^2$ — ideal 3D electric dipole scaling (reference only)')
ax.set_xlabel('$\\omega$ (dimensionless)')
ax.set_ylabel('Normalized power proxy')
ax.set_title('Frequency Scaling: Simulation vs Ideal 3D Dipole Reference\n'
             '(Deviations from $\\omega^4$ are expected for this 2D model)')
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### JC: In the future when I take EMII Im gonna explore if this value is correct and the frequency does scale at an alpha value of 2, but some of these values could be non physical due to the omega being 4 at the top end of the test, depending on the radius this could rotate faster than the speed of light. But in this case it is comfortably below it. at 0.4c

## Section 18 — Save Outputs to `fdtd_v4_results/`

Key results are saved to a local folder using **relative paths only**.

In [ ]:
OUTPUT_DIR = Path("fdtd_v4_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── 1. Main run summary JSON ──────────────────────────────────────────────────
summary = {
    "version":         "v5",
    "Nx_main":         Nx_main,
    "Ny_main":         Ny_main,
    "Lx_main":         Lx_main,
    "omega_main":      omega_main,
    "periods_main":    periods_main,
    "r_obs_main":      r_obs_main,
    "ring_radii_main": [float(r) for r in ring_radii_main],
    "boundary_method": result_main['boundary_method'],
    "nsteps_main":     nsteps_main,
    "dt_main":         dt_main,
    "cfl_number":      float(cfl_number),
    "cfl_number_2d":   float(CFL_number_main),
    "U_final":         float(result_main['energy_history'][-1]),
    "elapsed_s":       float(result_main['elapsed']),
    "mean_flux_avg":   float(mean_flux_avg),
    "std_flux_avg":    float(std_flux_avg),
    "med_cont_norm":   float(med_cont_norm),
    "med_gauss_rms":   float(med_gauss_rms),
    "c_measured":      float(c_measured) if not np.isnan(c_measured) else None,
    "alpha_freq_fit":  float(alpha_fit)  if not np.isnan(alpha_fit)  else None,
}
with open(OUTPUT_DIR / "main_run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved: main_run_summary.json")

# ── 1b. Time-series histories (NPZ) ───────────────────────────────────────────
flux_extras_keys = sorted(result_main['flux_extras'].keys())
flux_extras_arr  = np.stack([result_main['flux_extras'][r]
                             for r in flux_extras_keys], axis=0)
np.savez(
    OUTPUT_DIR / "main_run_histories.npz",
    t                  = result_main['t'],
    energy_history     = result_main['energy_history'],
    flux_history       = result_main['flux_out'],
    flux_signed        = result_main['flux_signed'],
    dipole_moment_history = np.stack([result_main['px'], result_main['py']], axis=0),
    flux_extras_radii  = np.array(flux_extras_keys),
    flux_extras        = flux_extras_arr,
    boundary_method    = np.array(result_main['boundary_method']),
    ring_radius        = np.array(r_obs_main),
)
print("Saved: main_run_histories.npz "
      "(energy_history, flux_history, dipole_moment_history, "
      "boundary_method, ring_radius)")

# ── 2. Convergence CSV ────────────────────────────────────────────────────────
if not conv_df.empty:
    conv_df.to_csv(OUTPUT_DIR / "convergence.csv", index=False)
    print("Saved: convergence.csv")
else:
    print("Convergence study was skipped (RUN_CONVERGENCE=False) — no CSV saved.")

# ── 3. Frequency scaling CSV ──────────────────────────────────────────────────
freq_df.to_csv(OUTPUT_DIR / "frequency_scaling.csv", index=False)
print("Saved: frequency_scaling.csv")

# ── 4. Key figures as PNG ─────────────────────────────────────────────────────
# Ring flux plot (auto-labeled with boundary method, omega, ring radius, grid)
fig_rf, ax_rf = plt.subplots(figsize=(8, 4))
ax_rf.plot(t_arr, flux_out, lw=1.2, label='Outward flux')
ax_rf.axvline(r_obs_main/c, color='k', ls='--', lw=1.0, label=f'$r_{{obs}}/c$')
ax_rf.axvline(t_avg_start,  color='orange', ls=':',  lw=1.0, label='Avg start')
ax_rf.axhline(mean_flux_avg, color='tab:green', lw=1.0, label=f'Mean={mean_flux_avg:.3e}')
ax_rf.set_xlabel('t'); ax_rf.set_ylabel('Flux'); ax_rf.legend(fontsize=9); ax_rf.grid(alpha=0.3)
ax_rf.set_title(f'Ring flux vs time — boundary={result_main["boundary_method"]}, '
                f'ω={omega_main}, r={r_obs_main}, N={Nx_main}')
fig_rf.tight_layout()
fig_rf.savefig(OUTPUT_DIR / "ring_flux_vs_time.png", dpi=120)
plt.close(fig_rf)
print("Saved: ring_flux_vs_time.png")

# Energy history plot
fig_eh, ax_eh = plt.subplots(figsize=(8, 4))
ax_eh.plot(result_main['t'], result_main['energy_history'], lw=1.2)
ax_eh.set_xlabel('t'); ax_eh.set_ylabel('Total field energy U(t)')
ax_eh.set_title(f'Energy history — boundary={result_main["boundary_method"]}, '
                f'ω={omega_main}, N={Nx_main}')
ax_eh.grid(alpha=0.3)
fig_eh.tight_layout()
fig_eh.savefig(OUTPUT_DIR / "energy_history.png", dpi=120)
plt.close(fig_eh)
print("Saved: energy_history.png")

# Frequency scaling plot
fig_fs, ax_fs = plt.subplots(figsize=(7, 5))
ax_fs.loglog(omegas_fit[valid], P_norm[valid], 'o-', label=f'Simulation α≈{alpha_fit:.2f}')
ax_fs.loglog(omegas_fit, ref_omega4, 's--', color='tab:orange', label='ω⁴ reference (ideal 3D)')
ax_fs.set_xlabel('ω'); ax_fs.set_ylabel('Normalized power proxy')
ax_fs.set_title(f'Frequency scaling — boundary={result_main["boundary_method"]}')
ax_fs.legend(); ax_fs.grid(True, which='both', alpha=0.3)
fig_fs.tight_layout()
fig_fs.savefig(OUTPUT_DIR / "frequency_scaling.png", dpi=120)
plt.close(fig_fs)
print("Saved: frequency_scaling.png")

# Last-step Bz snapshot
fig_bz, ax_bz = plt.subplots(figsize=(5.5, 5))
vlim = np.abs(result_main['Bz']).max()
ax_bz.imshow(result_main['Bz'], origin='lower', cmap='RdBu', vmin=-vlim, vmax=vlim,
             extent=[-Lx_main/2, Lx_main/2, -Ly_main/2, Ly_main/2])
ax_bz.set_title(f'$B_z$ at final step  '
                f'(boundary={result_main["boundary_method"]}, '
                f'ω={omega_main}, N={Nx_main})')
ax_bz.set_xlabel('x'); ax_bz.set_ylabel('y')
fig_bz.tight_layout()
fig_bz.savefig(OUTPUT_DIR / "bz_final_step.png", dpi=120)
plt.close(fig_bz)
print("Saved: bz_final_step.png")

print(f"\nAll outputs saved to: {OUTPUT_DIR.resolve()}")


---

## Optional Appendix — Long Parameter Sweep

> **This section does NOT execute by default.**  
> Set `RUN_LONG_SWEEP = True` only if you want to run the full joint parameter + resolution sweep.  
> **Runtime warning:** This could take many hours on a laptop.

The long sweep would:
1. Sample a large number of random $(\\omega, \text{sep}, a, b, \\phi)$ parameter combinations
2. Run each at moderate resolution ($N = 220$) for several periods
3. Score each trial by: residual cleanliness, outward flux consistency, and frequency-scaling coherence
4. Re-run the top candidates at higher resolutions ($N \in [220, 260, 300]$)
5. Report the best overall configuration and a convergence table

In [ ]:
RUN_LONG_SWEEP = True   # ← set True ONLY if you want the multi-hour sweep

if not RUN_LONG_SWEEP:
    print("RUN_LONG_SWEEP = False")
    print("Long parameter + resolution sweep is disabled.")
    print("To enable: set RUN_LONG_SWEEP = True in this cell.")
    print("\nThe sweep would:")
    print("  1. Sample N_trials random (omega, sep, a, b, phase) combinations")
    print("  2. Run each at N=220 for several periods")
    print("  3. Score by: flux level, continuity norm, Gauss RMS, freq-scaling coherence")
    print("  4. Re-run top candidates at N in [220, 260, 300]")
    print("  5. Report best configuration and save full results to fdtd_v4_results/long_sweep/")
else:
    # ── Minimal long-sweep skeleton ───────────────────────────────────────────
    import itertools
    rng = np.random.default_rng(seed=42)

    sweep_dir = OUTPUT_DIR / "long_sweep"
    sweep_dir.mkdir(exist_ok=True)

    N_trials     = 40      # adjust for desired depth
    omegas_ls    = [0.8, 1.2, 1.6, 2.0]
    resolutions  = [220, 260, 300]
    periods_ls   = 4.0
    r_obs_ls     = 3.2

    sweep_rows = []
    for trial in range(N_trials):
        om_ls  = float(rng.choice(omegas_ls))
        sep_ls = float(rng.uniform(0.6, 1.2))
        a_ls   = float(rng.uniform(0.25, 0.50))
        b_ls   = float(rng.uniform(0.15, 0.35))
        ph_ls  = float(rng.uniform(0.0, np.pi))

        # Run at base resolution
        N_base = 220
        dx_ls  = 10.0 / N_base
        dt_ls  = cfl_safety * dx_ls / c
        x_ls   = np.linspace(-5, 5, N_base, endpoint=False)
        y_ls   = np.linspace(-5, 5, N_base, endpoint=False)
        Xls, Yls = np.meshgrid(x_ls, y_ls, indexing='xy')

        T_ls      = 2*np.pi / om_ls
        nsteps_ls = int(np.ceil((periods_ls * T_ls + r_obs_ls) / dt_ls))
        t_avg_ls  = r_obs_ls + 1.5 * T_ls

        rl = run_fdtd(
            Xls, Yls, dx_ls, dx_ls, dt_ls,
            nsteps=nsteps_ls, omega=om_ls,
            Q=1.0, sep=sep_ls, a=a_ls, b=b_ls, phase=ph_ls,
            r_obs=r_obs_ls, ntheta_ring=180,
            save_interval=max(1, nsteps_ls//30), store_frames=False,
            continuity_interval=max(1, nsteps_ls//40),
            progress_ticks=False,
        )
        mask_ls  = rl['t'] >= t_avg_ls
        flux_ls  = float(np.mean(rl['flux_out'][mask_ls])) if mask_ls.any() else np.nan
        cont_ls  = float(np.nanmedian(rl['cont_norm']))
        gauss_ls = float(np.nanmedian(rl['gauss_rms']))

        # Simple score: higher flux, lower residuals
        score = flux_ls / (cont_ls + gauss_ls + 1e-15) if not np.isnan(flux_ls) else -np.inf

        sweep_rows.append({
            'trial': trial, 'omega': om_ls, 'sep': sep_ls, 'a': a_ls, 'b': b_ls,
            'phase': ph_ls, 'flux': flux_ls, 'cont_norm': cont_ls,
            'gauss_rms': gauss_ls, 'score': score,
        })
        print(f"Trial {trial+1:3d}/{N_trials}: ω={om_ls:.1f} score={score:.3e}")

    sweep_df = pd.DataFrame(sweep_rows).sort_values('score', ascending=False)
    sweep_df.to_csv(sweep_dir / "long_sweep_results.csv", index=False)
    print("\nLong sweep complete. Results saved to:", sweep_dir / "long_sweep_results.csv")
    display(sweep_df.head(10))

In [ ]:
from matplotlib.animation import FFMpegWriter, PillowWriter
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyArrowPatch
from pathlib import Path
import matplotlib.animation as animation

OUTPUT_DIR = Path("fdtd_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Animation parameters ──────────────────────────────────────────────────────
anim_fps        = 30
anim_duration_s = 15          # desired animation length in seconds
anim_total_frames = anim_fps * anim_duration_s

# Interpolate frame indices to get smooth, evenly-spaced frames
stored_n = len(result_main['frames_Bz'])
anim_frame_indices = np.round(
    np.linspace(0, stored_n - 1, anim_total_frames)
).astype(int)

ext = [-Lx_main/2, Lx_main/2, -Ly_main/2, Ly_main/2]
x_main_1d = X_main[0, :]
y_main_1d = Y_main[:, 0]

# ── Custom diverging colormaps with BLACK at zero (match Section 11) ────────
poynting_cmap = LinearSegmentedColormap.from_list(
    'poynting_black',
    ['#00ffff', '#0080ff', '#000000', '#ff8000', '#ffff00'],
    N=256,
)
efield_cmap = LinearSegmentedColormap.from_list(
    'efield_black',
    ['#ff00ff', '#8000ff', '#000000', '#00ff80', '#a8ff00'],
    N=256,
)

# Radial unit vectors (for signed radial scalars Sr, Er)
R_main = np.hypot(X_main, Y_main)
R_safe = np.where(R_main > 0, R_main, 1.0)
xhat   = X_main / R_safe
yhat   = Y_main / R_safe

# ── Pre-compute Poynting frames + global colour limits ────────────────────────
print("Computing Poynting frames and global colour limits …", end=' ', flush=True)
n_stored = len(result_main['frames_Bz'])
frames_Sx = np.empty_like(result_main['frames_Bz'])
frames_Sy = np.empty_like(result_main['frames_Bz'])
for k in range(n_stored):
    sx_k, sy_k = compute_poynting(
        result_main['frames_Ex'][k],
        result_main['frames_Ey'][k],
        result_main['frames_Bz'][k],
        mu0=mu0,
    )
    frames_Sx[k] = sx_k
    frames_Sy[k] = sy_k

# Save Poynting time-series alongside the existing frames_* arrays
result_main['frames_Sx'] = frames_Sx
result_main['frames_Sy'] = frames_Sy
np.savez(
    OUTPUT_DIR / "main_run_poynting_frames.npz",
    frame_t  = result_main['frame_t'],
    frames_Sx = frames_Sx,
    frames_Sy = frames_Sy,
    X = X_main, Y = Y_main,
    boundary_method = np.array(boundary_method_main),
    omega = np.array(omega_main),
    r_obs = np.array(r_obs_main),
)

# Global limits for diverging colormaps (use signed radial scalars + Bz)
sample_every = max(1, len(anim_frame_indices) // 60)
vlim_B_global  = 1e-12
vlim_Sr_global = 1e-12
vlim_Er_global = 1e-12
for i in anim_frame_indices[::sample_every]:
    Bz_i = result_main['frames_Bz'][i]
    Sr_i = frames_Sx[i] * xhat + frames_Sy[i] * yhat
    Er_i = result_main['frames_Ex'][i] * xhat + result_main['frames_Ey'][i] * yhat
    vlim_B_global  = max(vlim_B_global,  np.abs(Bz_i).max())
    vlim_Sr_global = max(vlim_Sr_global, np.abs(Sr_i).max())
    vlim_Er_global = max(vlim_Er_global, np.abs(Er_i).max())

print(f"done  (|Bz|_max={vlim_B_global:.3e}, "
      f"|Sr|_max={vlim_Sr_global:.3e}, |Er|_max={vlim_Er_global:.3e})")

# ── Figure setup ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.4))
fig.subplots_adjust(left=0.05, right=0.97, bottom=0.08, top=0.86, wspace=0.35)

# ── Initial frame ─────────────────────────────────────────────────────────────
fi0  = anim_frame_indices[0]
Bz0  = result_main['frames_Bz'][fi0]
Ex0  = result_main['frames_Ex'][fi0]
Ey0  = result_main['frames_Ey'][fi0]
Sx0  = frames_Sx[fi0]
Sy0  = frames_Sy[fi0]
Sr0  = Sx0 * xhat + Sy0 * yhat
Er0  = Ex0 * xhat + Ey0 * yhat

# Panel 0: Poynting flow (Sr, black-background diverging cmap)
axes[0].set_facecolor('black')
im_s = axes[0].imshow(Sr0, origin='lower', cmap=poynting_cmap,
                      vmin=-vlim_Sr_global, vmax=vlim_Sr_global, extent=ext)
cbar_s = fig.colorbar(im_s, ax=axes[0], fraction=0.046,
                      label=r'$S_r=\mathbf{S}\cdot\hat{\mathbf{r}}$')

# Panel 1: Electric flow (Er, black-background diverging cmap)
axes[1].set_facecolor('black')
im_e = axes[1].imshow(Er0, origin='lower', cmap=efield_cmap,
                      vmin=-vlim_Er_global, vmax=vlim_Er_global, extent=ext)
cbar_e = fig.colorbar(im_e, ax=axes[1], fraction=0.046,
                      label=r'$E_r=\mathbf{E}\cdot\hat{\mathbf{r}}$')

# Panel 2: Bz field with Poynting streamlines (RdBu)
im_bz = axes[2].imshow(Bz0, origin='lower', cmap='RdBu',
                       vmin=-vlim_B_global, vmax=vlim_B_global, extent=ext)
cbar_bz = fig.colorbar(im_bz, ax=axes[2], fraction=0.046, label='$B_z$')

for ax, lbl in zip(axes, [
    'Poynting flow  (orange = outward, cyan = inward)',
    'Electric flow  (green = outward, magenta = inward)',
    '$B_z$ field  with Poynting streamlines',
]):
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title(lbl, fontsize=11)
    ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3])
    ax.set_aspect('equal')

# Observation ring overlay (one per panel)
for ax in axes:
    ax.add_patch(plt.Circle((0, 0), r_obs_main, fill=False,
                            color='lime', lw=1.0, ls='--', alpha=0.7))

title_txt = fig.suptitle('', fontsize=12)

# Source-position markers (updated each frame, all three panels)
def _make_marker(ax, edge):
    return ax.scatter([], [], s=60, marker='o', facecolors='none',
                      edgecolors=edge, lw=1.5, zorder=5)

scat_pos_s  = _make_marker(axes[0], 'red')
scat_neg_s  = _make_marker(axes[0], 'blue')
scat_pos_e  = _make_marker(axes[1], 'red')
scat_neg_e  = _make_marker(axes[1], 'blue')
scat_pos_bz = _make_marker(axes[2], 'red')
scat_neg_bz = _make_marker(axes[2], 'blue')

# ── Streamline helpers (streamplot has no in-place update; redraw each frame) ─
stream_density = 1.3
stream_lw      = 0.7
stream_arrow   = 1.0
# Track streamplot artists per axis so they can be removed each frame
stream_artists = {0: [], 1: [], 2: []}

def _clear_streamlines(ax_idx):
    """Remove previously drawn streamlines (lines + arrows) on this axis."""
    ax = axes[ax_idx]
    for art in stream_artists[ax_idx]:
        try:
            art.remove()
        except (ValueError, NotImplementedError):
            pass
    stream_artists[ax_idx] = []
    # Streamplot leaves FancyArrowPatch objects in ax.patches; strip them
    # but keep our observation-ring Circle.
    for p in list(ax.patches):
        if isinstance(p, FancyArrowPatch):
            p.remove()

def _draw_streamlines(ax_idx, Fx, Fy, color):
    sp = axes[ax_idx].streamplot(
        x_main_1d, y_main_1d, Fx, Fy,
        color=color, density=stream_density,
        linewidth=stream_lw, arrowsize=stream_arrow,
    )
    stream_artists[ax_idx].append(sp.lines)

# Draw streamlines for the initial frame
_draw_streamlines(0, Sx0, Sy0, 'white')   # Poynting flow
_draw_streamlines(1, Ex0, Ey0, 'white')   # Electric flow
_draw_streamlines(2, Sx0, Sy0, 'black')   # Bz panel: Poynting streamlines

# ── Animation update ─────────────────────────────────────────────────────────
def update(frame_num):
    fi   = anim_frame_indices[frame_num]
    t_fr = result_main['frame_t'][fi]

    Bz_fr = result_main['frames_Bz'][fi]
    Ex_fr = result_main['frames_Ex'][fi]
    Ey_fr = result_main['frames_Ey'][fi]
    Sx_fr = frames_Sx[fi]
    Sy_fr = frames_Sy[fi]
    Sr_fr = Sx_fr * xhat + Sy_fr * yhat
    Er_fr = Ex_fr * xhat + Ey_fr * yhat

    im_s.set_data(Sr_fr)
    im_e.set_data(Er_fr)
    im_bz.set_data(Bz_fr)

    # Redraw streamlines (streamplot has no in-place UVC update)
    for k in (0, 1, 2):
        _clear_streamlines(k)
    _draw_streamlines(0, Sx_fr, Sy_fr, 'white')
    _draw_streamlines(1, Ex_fr, Ey_fr, 'white')
    _draw_streamlines(2, Sx_fr, Sy_fr, 'black')

    # Update source markers on all three panels
    xs, ys, signs = source_positions(t_fr, omega_main, sep_main,
                                      source_type=SOURCE_TYPE,
                                      n_ring=N_ring, ring_r=ring_radius)
    pos_xy = np.column_stack([xs[signs > 0], ys[signs > 0]]) if (signs > 0).any() else np.empty((0, 2))
    neg_xy = np.column_stack([xs[signs < 0], ys[signs < 0]]) if (signs < 0).any() else np.empty((0, 2))

    scat_pos_s.set_offsets(pos_xy);  scat_neg_s.set_offsets(neg_xy)
    scat_pos_e.set_offsets(pos_xy);  scat_neg_e.set_offsets(neg_xy)
    scat_pos_bz.set_offsets(pos_xy); scat_neg_bz.set_offsets(neg_xy)

    title_txt.set_text(
        f'Rotating Amorphous Dipole — 2D FDTD  |  '
        f't = {t_fr:.3f}  (frame {frame_num+1}/{anim_total_frames})\n'
        f'ω={omega_main}, N={Nx_main}×{Ny_main}, boundary={boundary_method_main}'
    )
    # blit=False, so just return None
    return ()

ani = animation.FuncAnimation(fig, update, frames=anim_total_frames,
                               interval=1000 / anim_fps, blit=False)

# ── Save ──────────────────────────────────────────────────────────────────────
anim_path_gif = OUTPUT_DIR / "fdtd_animation5.gif"
anim_path_mp4 = OUTPUT_DIR / "fdtd_animation5.mp4"

print(f"Saving GIF  → {anim_path_gif}  ({anim_total_frames} frames @ {anim_fps} fps) …")
ani.save(str(anim_path_gif), writer=PillowWriter(fps=anim_fps), dpi=110)
print("GIF saved ✓")

try:
    print(f"Saving MP4  → {anim_path_mp4} …")
    ani.save(str(anim_path_mp4),
             writer=FFMpegWriter(fps=anim_fps, bitrate=2400,
                                 extra_args=['-vcodec', 'libx264', '-pix_fmt', 'yuv420p']),
             dpi=110)
    print("MP4 saved ✓")
except Exception as e:
    print(f"MP4 save failed (ffmpeg not available?): {e}")
    print("GIF is still available.")

plt.close(fig)
print(f"\nAnimation complete — {anim_total_frames} frames, {anim_duration_s}s @ {anim_fps} fps")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Saved Poynting frames → {OUTPUT_DIR / 'main_run_poynting_frames.npz'}")
